# RSNA Knee: rank blend of the public Raptor model and our compact model (S6)

Private. Stage 1 runs the public notebook [RSNA Knee 0949 Anatomical Mirror](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror) as published: its checkpoint check and its scored source (SHA-256 `bcd864cb...`, verified when this notebook was built), with two marked `[blend]` changes (output file name, raw-probability dump). Stage 2 runs our compact convnext_small model S6 (4 fold models, arm `E_lrprior`, bundle `209a32c8b50a6b1c`) unchanged. Stage 3 averages per-finding percentile ranks with weights fixed before any run (Raptor 0.7, ours 0.3); nothing is fitted on labels or on the test set.


## Credits and required acknowledgement

Built on dreaddevelopment's [Raptor CoAtNet/MIL inference recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). These are our retrained checkpoints, rather than those baseline weights. Weak-label sources include [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), combined with our report-derived labels. Source reports and study-level label tables are not included here. The CoAtNet backbone comes from timm. Competition data is provided under the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study](https://nda.nih.gov/study.html?id=3407) 路 [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).


In [ ]:
# Fork954 --s6-anchor: start clock for our arm's time budget (added by src/build_fork954.py). Stage 1 (the OAI-trained
# CoAtNet) is not in this notebook, so its clock NB_T0, which stage 2 reads (T0 = NB_T0), is set here instead.
import time as _fork_time0
_FORK_T0 = _fork_time0.time()
NB_T0 = _FORK_T0


In [ ]:
ARM = 'E_lrprior'
BUNDLE_ID = '209a32c8b50a6b1c'
DEADLINE_HOURS = 8.3
CODE_SLUG = 'rsna-orthofoundation-fold0-epoch7-20261004'
COMPETITION = 'rsna-knee-abnormality-detection'
CODE_SHA = {'native_cache.py': 'c53ce8985406b3b918d9e821d43518479e89d9f946daebd817f378e3949718b9', 'native_data.py': 'f59afe205c8e193369fa3ccd22edcce8df3b293205fc48ddb1dd45169ee43b36', 'input_provenance.py': 'b3a22500c7f976f6ee040239ada5cacecf2326861cb49af1c7d8cfdeeb55fcda', 'intensity_mapping.py': '19be2d8cbd6f8971c1288d7512a8cca41d143470c2ba6e9b9096fde035fa9d41', 'coronal_views.py': '841f216d2682ae2c0f8afd82fc902b9fa74924d7145b952069dea8b0bbd03c40', 'exact_duplicates.py': '51b38db21ce9c5fe2b6278a835983414756b61ac6af81aba380b44b41b075897', 'dicom_geometry.py': '12aa501f9704aef15b7a3bbd560951acb6d95b89b7de0141d6cf89fc3f051ab1', 'common.py': '8283fa1f8e7ef8cfacbba52df1715636be3c9ad688e39912814e7d5bcf57105f'}
SOURCE_SHA = {'px_cache.py': 'b13e38a7ad3bd496f8027e02209653e224b0926db4f0853eeb0d076a2d6e2632', 'compact_train.py': '14767c7a8f0daa986477511fe9a83aefe032a208ae978fabf15a5f896992ac80', 'compact_infer.py': '34f433fe72b732543813165c3397ec80cef01b0d49b25b7b22668714f0b0ef4a', 'probe_infer.py': 'a6cc31448019c84be5868d0a2ff1d99719b23b0f86110efc03efff3ce6bd6980', 'probe_dual.py': '9171522ada7ad94145529bb0a47ec48b6f0d8d32d651a60f3f207558649e36b8', 'knee_anchor_bank_v2.py': 'e564807a129dfcd3c984348917436ee6083066a4d685340587ebd1d3ea14f309', 'side_common.py': 'bbcc5d027834e6a6fd48c4a8cf228e25661b7522c792587c840fe4d7cd5c0566'}
PROBE_CLASS = None
C_SLUG = 'rsna-knee-compact-v1'
W_SLUG = 'rsna-knee-ofl-ssl-weights'
SOURCES = {'px_cache.py': '"""Per-slice pixel cache on the fast disk, for training compact image models without re-reading F:.\n\nData preparation only: no model, no optimizer.  Reads F:/rsnaknee/cache_native (read-only) and writes\nONLY under --out:\n  px.u8.npy    [N, S, S] uint8   one row per (series, view, slice), slices in physical order\n  meta.f32.npy [N, 12]\n  index.json   study -> [start, n_rows, n_series], per-series blocks, config, progress\nEach row is exactly one channel of the mainline\'s validation-mode window (same crop geometry, same\nINTER_AREA resize, same series_companded intensity mapping), at S pixels instead of 384.\nViews: full field + 100 mm local; coronal uses the mainline\'s bilateral views.\n"""\nimport sys, os, json, time, hashlib, argparse\nsys.dont_write_bytecode = True\nCODE_ROOT = os.environ.get("RSNA_CODE_ROOT", "D:/rsnaknee")      # mainline checkout (Linux box: /root/autodl-tmp/rsnaknee)\nsys.path.insert(0, CODE_ROOT)\nimport numpy as np\nimport torch\n\nLOCAL_MM = 100.\nTHIN_MM, SLAB_MM = 1.75, 3.5       # stacks thinner than THIN_MM are averaged into SLAB_MM slabs (see StudySlices.build)\nPOLICY = "series_companded"\nPLANE_ID = {"Sagittal": 0, "Coronal": 1, "Axial": 2}\nMETA_COLS = ["series_idx", "slot", "plane", "view_id", "rel_pos", "log_spacing", "extent_130", "fluid",\n             "fatsat", "slice_order", "n_slices", "real_series_idx"]\nUIDK = "StudyInstanceUID"\n\n\ndef study_json(cache, uid):\n    return os.path.join(cache, "studies", hashlib.sha256(uid.encode()).hexdigest()[:24] + ".json")\n\n\n# View sets.  v1: full field + 100 mm local at the field centre (coronal: bilateral locals).  v2 adds, for sagittal and axial\n# series, a 100 mm "patellofemoral" local (view id 4; ids 2/3 are the coronal bilateral locals) centred anterior-superior: in the canonical cache anterior is +column\n# for sagittal and -row for axial, and a centre-of-field local cuts the patella at its edge in every study inspected\n# (10-06; PF OA is the class with the largest report-vs-image gap on the 58 official studies, 0.92 vs 0.82).\n# Centre tuples are (row fraction, column fraction) as view_geometry expects.\nPF_CENTER = {"Sagittal": (.38, .60), "Axial": (.38, .50)}\n# v3 (high-resolution set, meant for a 384 px cache): NO full-field view; sagittal/axial = centre local (1) + patellofemoral\n# local (4) + posterior local (5: popliteal fossa / Baker\'s cyst / PCL; posterior = -column for sagittal, +row for axial);\n# coronal = the two bilateral locals only.  Three overlapping 100 mm locals cover what the 1.8 px/mm full view showed.\nPOST_CENTER = {"Sagittal": (.55, .35), "Axial": (.62, .50)}\n\n\ndef knee_anchor(vol, plane):\n    """(row_frac, col_frac) of the knee in the series\' in-plane frame from the foreground of the 3 middle slices.\n    Falls back to the field centre when the leg fills the field or nothing is found."""\n    import cv2\n    m = len(vol) // 2\n    mid = np.asarray(vol[max(0, m - 1):m + 2]).astype(np.float32).mean(0)\n    H, W = mid.shape\n    hi = float(np.percentile(mid, 99.5))\n    if hi <= 0:\n        return (.5, .5), "empty"\n    fg = (mid > 0.08 * hi).astype(np.uint8)\n    n, lab, stats, _ = cv2.connectedComponentsWithStats(fg, 8)\n    if n <= 1:\n        return (.5, .5), "no_fg"\n    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n    x, y, w, h = [int(v) for v in stats[k, :4]]\n    if w >= 0.97 * W and h >= 0.97 * H:\n        return (.5, .5), "fills_field"\n    comp = (lab == k)\n    if plane in ("Sagittal", "Coronal"):\n        width = comp[y:y + h].sum(1).astype(np.float32)\n        if len(width) >= 15:\n            kern = np.ones(15, np.float32) / 15\n            width = np.convolve(width, kern, mode="same")\n        row = (y + int(np.argmax(width))) / max(1, H - 1)\n    else:\n        row = (y + h / 2.) / max(1, H - 1)\n    col = (x + w / 2.) / max(1, W - 1)\n    return (float(row), float(col)), "ok"\n\n\ndef anchored(anchor, spacing_mm, shape, d_row_mm, d_col_mm):\n    """centre = anchor shifted by physical offsets (rows: +down, cols: +right), as (row_frac, col_frac)."""\n    sy, sx = float(spacing_mm[0]), float(spacing_mm[1])\n    H, W = shape\n    return (min(1., max(0., anchor[0] + d_row_mm / (sy * max(1, H - 1)))),\n            min(1., max(0., anchor[1] + d_col_mm / (sx * max(1, W - 1)))))\n\n\ndef series_views(plane, native_shape, spacings, size, views="v1", anchor=None, spacing_mm=None, patella=None):\n    if views in ("v3k", "v4"):\n        # knee-centred set from the anchor bank (SKM-TEA joint centre projected through the geometry bank):\n        # v4 adds the full field of view (id 0) of every series in front of the locals, so nothing is out of sight.\n        full = [(0, None, (.5, .5))] if views == "v4" else []\n        # centre local (1) at the joint; PF local (4) at the patella anchor when the segmentation found it, else\n        # anterior-superior of the joint; posterior local (5); coronal medial (2) / lateral (3) 30 mm either side.\n        # Anterior = +col (sagittal), -row (axial) in the canonical cache.\n        shp = tuple(native_shape[-2:])\n        a = anchor if anchor is not None else (.5, .5)\n        sp = spacing_mm if spacing_mm is not None else (1., 1.)\n        if plane == "Sagittal":\n            pf = patella if patella is not None else anchored(a, sp, shp, -25., +40.)   # patella centre ~40 mm anterior of the joint centre (gold montages: +30 cut it)\n            return full + [(1, LOCAL_MM, a), (4, LOCAL_MM, pf), (5, LOCAL_MM, anchored(a, sp, shp, +5., -35.))]\n        if plane == "Axial":\n            pf = patella if patella is not None else anchored(a, sp, shp, -40., 0.)\n            return full + [(1, LOCAL_MM, a), (4, LOCAL_MM, pf), (5, LOCAL_MM, anchored(a, sp, shp, +35., 0.))]\n        return full + [(2, LOCAL_MM, anchored(a, sp, shp, 0., -30.)), (3, LOCAL_MM, anchored(a, sp, shp, 0., +30.))]\n    if plane == "Coronal":\n        if views == "v2add":\n            return []\n        from training.coronal_views import coronal_bilateral_views\n        out = coronal_bilateral_views(tuple(native_shape[-2:]), spacings, size, LOCAL_MM)\n        if views == "v3":                     # bilateral locals only (view id 0 is the full field)\n            return [v for v in out if int(v[0]) != 0]\n        return out\n    if views == "v2add":                      # add-on cache: ONLY the rows v2 adds to v1 (coronal adds none)\n        return [(4, LOCAL_MM, PF_CENTER[plane])] if plane in PF_CENTER else []\n    if views == "v3":\n        return [(1, LOCAL_MM, (.5, .5)), (4, LOCAL_MM, PF_CENTER[plane]), (5, LOCAL_MM, POST_CENTER[plane])]\n    out = [(0, None, (.5, .5)), (1, LOCAL_MM, (.5, .5))]\n    if views == "v2":\n        out.append((4, LOCAL_MM, PF_CENTER[plane]))\n    elif views != "v1":\n        raise ValueError("unknown view set " + str(views))\n    return out\n\n\nclass StudySlices(torch.utils.data.Dataset):\n    def __init__(self, cache, uids, size, views="v1", anchors=None):\n        self.cache, self.uids, self.size, self.views = cache, uids, size, views\n        self.anchors = anchors                   # directory of <sha24(study)>.json from knee_anchor_bank_v2.py (views v3k)\n        self._resolver = None\n\n    def __len__(self):\n        return len(self.uids)\n\n    def __getitem__(self, i):\n        uid = self.uids[i]\n        try:\n            return self.build(uid)\n        except Exception as exc:\n            return {"uid": uid, "error": repr(exc)[:400], "px": torch.zeros((0, self.size, self.size), dtype=torch.uint8),\n                    "meta": torch.zeros((0, len(META_COLS)), dtype=torch.float32), "blocks": [], "removed": 0}\n\n    def build(self, uid):\n        import cv2\n        cv2.setNumThreads(1)\n        from training.input_provenance import view_geometry\n        from training.intensity_mapping import normalize_intensity\n        from training.native_data import select_display_window\n        from training.exact_duplicates import ExactDuplicateResolver\n        if self._resolver is None:\n            self._resolver = ExactDuplicateResolver(None, None, False)\n        S = self.size\n        with open(study_json(self.cache, uid), encoding="utf-8") as fh:\n            study = json.load(fh)\n        # study JSONs carry the builder\'s (Windows) path of every series: relocate to this cache before anything reads them;\n        # a series the native cache never rendered (12 studies / 31 series) is dropped so the study keeps its other series\n        series, skipped = [], []\n        for s in study["series"]:\n            path = s["path"]\n            if not os.path.exists(path):\n                path = os.path.join(self.cache, "series", os.path.basename(path.replace(chr(92), "/")))\n            if not os.path.exists(path):\n                skipped.append(s["SeriesInstanceUID"]); continue\n            series.append(dict(s, path=path))\n        if not series:\n            raise ValueError("no series file present for this study (%d missing)" % len(skipped))\n        study = dict(study, series=series)\n        plan = self._resolver.resolve(study)\n        self._resolver.cache.clear()\n        px, meta, blocks = [], [], []\n        for real_idx in plan["representative_indices"]:\n            s = study["series"][real_idx]\n            path = s["path"]\n            volume = np.load(path, mmap_mode="r", allow_pickle=False)\n            pos = np.asarray(s["positions"], float)\n            if volume.ndim != 3 or len(pos) != len(volume) or not len(pos) or not np.isfinite(pos).all():\n                raise ValueError("native volume/slice-position mapping is incomplete")\n            if s["plane"] == "Sagittal" and s.get("side") == "R":\n                pos = -pos\n            order = np.argsort(pos, kind="stable")\n            pos = pos[order]\n            spacing = float(np.median(np.diff(pos))) if len(pos) > 1 else 0.\n            window = select_display_window(s, POLICY)\n            fluid = s.get("fluid_sensitive", -1)\n            fat = s.get("fat_suppression", -1)\n            sidx = len(blocks)\n            vol = np.asarray(volume[order])                      # slices in physical order, read once\n            order_rep = order                                     # native slice index behind each stored slice (in-plane spacing lookups)\n            if 0 < spacing < THIN_MM:\n                # thin 3-D stacks (0.4-1.5 mm): average groups of slices into ~SLAB_MM slabs, so every slice contributes\n                # (a per-slice window sampler at a physical stride would skip ~90 % of a 0.4 mm stack) and the series\n                # carries the same slab statistics as a 2-D acquisition.  Same code at Kaggle inference (StudySlices).\n                g = max(1, int(round(SLAB_MM / spacing)))\n                n_new = len(pos) // g + (1 if len(pos) % g >= max(1, g // 2) else 0)\n                if n_new >= 2:\n                    chunks = [vol[j * g:min(len(vol), (j + 1) * g)] for j in range(n_new)]\n                    vol = np.stack([np.rint(c.astype(np.float32).mean(0)).astype(vol.dtype) for c in chunks])\n                    pos = np.array([pos[j * g:min(len(pos), (j + 1) * g)].mean() for j in range(n_new)])\n                    order_rep = np.array([order[j * g] for j in range(n_new)])\n                    spacing = float(np.median(np.diff(pos)))\n            block = {"series": s["SeriesInstanceUID"], "plane": s["plane"], "n_slices": int(len(pos)), "views": []}\n            anchor = pat = None\n            sp0 = None\n            if self.views in ("v3k", "v4"):\n                if self.anchors:\n                    ap_ = os.path.join(self.anchors, hashlib.sha256(uid.encode()).hexdigest()[:24] + ".json")\n                    ent = (json.load(open(ap_)).get("series", {}) if os.path.exists(ap_) else {}).get(s["SeriesInstanceUID"])\n                    if ent and ent.get("in_field", True):\n                        anchor = tuple(float(v) for v in ent["joint"])\n                        pat = tuple(float(v) for v in ent["patella"]) if ent.get("patella") else None\n                if anchor is None:\n                    anchor = (.5, .5)                                # no anchor entry: field centre (the foreground heuristic was worse on g02)\n                sp0 = s["spacing_mm"][int(order_rep[0])]\n            for view_id, crop, center_xy in series_views(s["plane"], volume.shape, s["spacing_mm"], S, self.views, anchor, sp0, pat):\n                block["views"].append([int(view_id), len(px)])\n                for k in range(len(pos)):\n                    sp = s["spacing_mm"][int(order_rep[k])]\n                    g = view_geometry(vol.shape[1:], sp, S, crop_mm=crop, center=center_xy)\n                    x, y, x1, y1 = g.crop_xyxy\n                    pxo, pyo, _, _ = g.valid_xyxy\n                    oh, ow = g.resized_shape\n                    resized = cv2.resize(vol[k, y:y1, x:x1].astype(np.float32), (ow, oh), interpolation=cv2.INTER_AREA)\n                    resized = normalize_intensity(resized, window, POLICY)\n                    out = np.zeros((S, S), np.uint8)\n                    out[pyo:pyo + oh, pxo:pxo + ow] = np.rint(np.clip(resized, 0, 1) * 255).astype(np.uint8)\n                    px.append(out)\n                    relative = 2 * (pos[k] - pos[0]) / max(pos[-1] - pos[0], 1e-6) - 1\n                    meta.append([sidx, s["slot"], PLANE_ID[s["plane"]], view_id, relative, np.log1p(abs(spacing)),\n                                 g.extent / 130., -1 if fluid is None else fluid, -1 if fat is None else fat,\n                                 k, len(pos), real_idx])\n            blocks.append(block)\n            del volume, vol\n        if px:\n            px = torch.from_numpy(np.stack(px))\n            meta = torch.tensor(meta, dtype=torch.float32)\n        else:\n            px = torch.zeros((0, S, S), dtype=torch.uint8)\n            meta = torch.zeros((0, len(META_COLS)), dtype=torch.float32)\n        return {"uid": uid, "error": None, "px": px, "meta": meta, "blocks": blocks, "skipped_series": skipped,\n                "removed": int(plan.get("removed_series", 0))}\n\n\ndef count_upper(cache, uid, size, views="v1"):\n    with open(study_json(cache, uid), encoding="utf-8") as fh:\n        study = json.load(fh)\n    n = 0\n    for s in study["series"]:\n        try:\n            v = len(series_views(s["plane"], s["native_shape"], s["spacing_mm"], size, "v3" if views in ("v3k", "v4") else views)) + (1 if views == "v4" else 0)\n        except Exception:\n            v = 3\n        n += v * len(s["positions"])\n    return n\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--cache", default="F:/rsnaknee/cache_native")\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--first", default=None)\n    ap.add_argument("--size", type=int, default=288)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--workers", type=int, default=10)\n    ap.add_argument("--seed", type=int, default=20261005)\n    ap.add_argument("--margin-gb", type=float, default=60.)\n    ap.add_argument("--anchors", default=None, help="anchor bank directory (knee_anchor_bank.py) for --views v3k")\n    ap.add_argument("--views", default="v1", choices=["v1", "v2", "v2add", "v3", "v3k", "v4"],\n                    help="v2 = v1 + patellofemoral local view for sagittal/axial; v2add = only that added view (an add-on cache "\n                         "that compact_train reads next to a v1 cache with --px-extra)")\n    a = ap.parse_args()\n    os.makedirs(a.out, exist_ok=True)\n    t0 = time.time()\n    import pandas as pd\n    uids = pd.read_csv(os.path.join(a.cache, "patients.csv"), dtype=str)[UIDK].tolist()\n    known = set(uids)\n    first = json.load(open(a.first, encoding="utf-8")) if a.first else []\n    first = [u for u in first if u in known]\n    rest = sorted(known - set(first))\n    rng = np.random.default_rng(a.seed)\n    rest = [rest[i] for i in rng.permutation(len(rest))]\n    order = first + rest\n    if a.limit:\n        order = order[:a.limit]\n    index_path = os.path.join(a.out, "index.json")\n    if os.path.exists(index_path):\n        index = json.load(open(index_path, encoding="utf-8"))\n        if index["order"] != order or index["config"]["size"] != a.size or index["config"].get("views", "v1") != a.views:\n            raise SystemExit("existing index was built for another order/size/views; use a new --out")\n    else:\n        counts = [count_upper(a.cache, u, a.size, a.views) for u in order]\n        index = {"order": order, "n_upper": int(sum(counts)), "done": {}, "blocks": {}, "cursor": 0, "errors": {},\n                 "removed_series": 0, "meta_cols": META_COLS,\n                 "config": {"size": a.size, "local_mm": LOCAL_MM, "policy": POLICY,\n                            "views": a.views, "anchors": a.anchors,\n                            "view_recipe": "global_local+coronal_bilateral" + ("+pf_local" if a.views == "v2" else ""),\n                            "duplicates": "collapse_exact",\n                            "rows": "one per (series, view, slice), physical order", "gold_first": len(first)}}\n        json.dump(index, open(index_path, "w", encoding="utf-8"))\n    N = max(1, index["n_upper"])\n    gb = N * a.size * a.size / 2 ** 30\n    print("studies", len(order), "row upper bound", N, "pixel file GB %.1f" % gb, "count seconds %.1f" % (time.time() - t0), flush=True)\n    import shutil\n    free = shutil.disk_usage(a.out).free / 2 ** 30\n    if not os.path.exists(os.path.join(a.out, "px.u8.npy")) and free < gb + a.margin_gb:\n        raise SystemExit("not enough free space: need %.0f GB + %.0f GB margin, have %.0f GB" % (gb, a.margin_gb, free))\n\n    def open_mm(path, shape, dtype):\n        if os.path.exists(path):\n            mm = np.load(path, mmap_mode="r+")\n            if tuple(mm.shape) != tuple(shape):\n                raise SystemExit("existing memmap has another shape: " + path)\n            return mm\n        return np.lib.format.open_memmap(path, mode="w+", dtype=dtype, shape=shape)\n\n    px_mm = open_mm(os.path.join(a.out, "px.u8.npy"), (N, a.size, a.size), np.uint8)\n    meta_mm = open_mm(os.path.join(a.out, "meta.f32.npy"), (N, len(META_COLS)), np.float32)\n    todo = [u for u in order if u not in index["done"] and u not in index["errors"]]\n    loader = torch.utils.data.DataLoader(StudySlices(a.cache, todo, a.size, a.views, a.anchors), batch_size=None, shuffle=False,\n                                         num_workers=a.workers, prefetch_factor=2 if a.workers else None,\n                                         persistent_workers=bool(a.workers))\n    state = {"t": time.time(), "studies": 0, "last": time.time()}\n\n    def save(final=False):\n        px_mm.flush(); meta_mm.flush()\n        tmp = index_path + ".tmp"\n        json.dump(index, open(tmp, "w", encoding="utf-8"))\n        os.replace(tmp, index_path)\n        el = time.time() - state["t"]\n        sps = state["studies"] / max(el, 1e-6)\n        prog = {"done_studies": len(index["done"]), "errors": len(index["errors"]), "total_studies": len(order),\n                "rows_written": index["cursor"], "elapsed_s": round(el, 1), "studies_per_s": round(sps, 3),\n                "eta_min": round((len(todo) - state["studies"]) / max(sps, 1e-6) / 60, 1), "final": final,\n                "time": time.strftime("%Y-%m-%d %H:%M:%S")}\n        json.dump(prog, open(os.path.join(a.out, "progress.json"), "w"))\n        print(json.dumps(prog), flush=True)\n\n    for item in loader:\n        uid = item["uid"]\n        state["studies"] += 1\n        if item["error"]:\n            index["errors"][uid] = item["error"]\n            continue\n        n = len(item["px"])\n        cur = index["cursor"]\n        if cur + n > N:\n            raise RuntimeError("row count exceeds the allocated upper bound")\n        if n:\n            px_mm[cur:cur + n] = item["px"].numpy()\n            meta_mm[cur:cur + n] = item["meta"].numpy()\n        for b in item["blocks"]:\n            b["views"] = [[v, cur + off] for v, off in b["views"]]\n        index["done"][uid] = [cur, n, len(item["blocks"])]\n        index["blocks"][uid] = item["blocks"]\n        index["cursor"] = cur + n\n        index["removed_series"] += item["removed"]\n        if time.time() - state["last"] > 90:\n            save(); state["last"] = time.time()\n    save(final=True)\n    print("DONE rows", index["cursor"], "studies", len(index["done"]), "errors", len(index["errors"]),\n          "minutes %.1f" % ((time.time() - t0) / 60), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'compact_train.py': '"""Compact fine-tuned image model on the per-slice pixel cache.  The user authorised fine-tuning a compact model on 2026-10-05 evening\n(the mainline OrthoFoundation training hold on the 5090 is untouched).  v1.1: fold models are saved.\n\nSame contract as probe_train.py (so probe_eval.py and fuse_eval.py read its outputs unchanged):\n  - never reads official labels; gold studies are never trained on; targets come from the same CSVs\n  - K-fold over the non-gold pool with the same hash folds; writes gold predictions (mean of K models)\n    and out-of-fold predictions for every pool study\n  - arms: S_strict, P_presence, E_prior, E_lrprior, or name=bce:<column> / name=lr:<column> from --extra\n\nModel: 2.5D windows (three physically adjacent slices as channels) -> CNN encoder -> per-class attention\nover the windows of a study -> 12 logits.  Training samples a fixed number of windows per study; evaluation\nuses every third slice as a centre (the mainline\'s cover sampling) in both views.\n"""\nimport argparse, contextlib, csv, hashlib, json, math, os, sys, time\nsys.dont_write_bytecode = True\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",\n          "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nC = len(LABELS)\n\n\nPREDICT_CHUNK = int(os.environ.get("CT_PREDICT_CHUNK", "512"))   # v7: evaluation windows per encoder call (512 = v6)\n\n\ndef covering_centers(n, phase=1, stride=3):\n    if stride <= 1:\n        return list(range(n))\n    c = list(range(min(phase, n - 1), n, stride))\n    if c[0] > 1:\n        c.insert(0, 0)\n    if c[-1] < n - 2:\n        c.append(n - 1)\n    return c\n\n\ndef phys_step(spacing_mm, unit_mm):\n    """slices per ~unit_mm of anatomy: 1 for an ordinary 3-4 mm series, ~9 for a 0.4 mm 3-D stack (unit_mm <= 0: always 1)."""\n    if unit_mm <= 0:\n        return 1\n    return max(1, int(round(unit_mm / max(float(spacing_mm), 0.1))))\n\n\ndef window_rows(blocks, meta_of, mode, slices, eval_stride=3, cover_stride=3, phys_mm=0.0, rng=None, slice_jitter=0.0,\n                train_sampling="cover"):\n    """The window list of one study.  blocks: [{\'n_slices\': n, \'views\': [(file, view_id, start), ...]}]; meta_of(file, row)\n    -> the px meta row of that window (column 5 = log1p(slice spacing in mm)).  Returns tuples\n    (file, centre row, row of slice -r, ..., row of slice +r).  \'train\' covers every (series, view) at stride\n    cover_stride x phys_step with a random phase (and takes the K neighbours two steps apart with probability slice_jitter);\n    \'eval\' covers at stride eval_stride x phys_step.  phys_mm = 0 reproduces the old per-slice behaviour."""\n    cand = []\n    r_ = slices // 2\n    for b in blocks:\n        n = b["n_slices"]\n        for f, view_id, start in b["views"]:\n            ps = phys_step(math.expm1(float(meta_of(f, start)[5])), phys_mm)\n            if mode == "train":\n                if train_sampling == "cover":\n                    st = cover_stride * ps\n                    ks = covering_centers(n, int(rng.integers(0, st)), st)\n                else:\n                    ks = range(n)\n            else:\n                ks = covering_centers(n, 1, eval_stride * ps)\n            for k in ks:\n                step_ = ps * (2 if (mode == "train" and slice_jitter > 0 and rng.random() < slice_jitter) else 1)\n                cand.append((f, start + k) + tuple(start + min(n - 1, max(0, k + d * step_)) for d in range(-r_, r_ + 1)))\n    return cand\n\n\nclass Studies(torch.utils.data.Dataset):\n    """Returns uint8 windows [W,3,S,S] and meta [W,6] for one study.  mode \'train\' samples, \'eval\' covers."""\n\n    def __init__(self, px_dir, uids, n_win, mode, seed, eval_stride=3, train_sampling="uniform", cover_stride=3, px_extra=None, slices=3,\n                 slice_jitter=0.0, phys_mm=0.0, px_override=None, extent_meta=False):\n        self.dir, self.uids, self.n_win, self.mode, self.seed = px_dir, uids, n_win, mode, seed\n        self.extent_meta = bool(extent_meta)\n        self.eval_stride, self.train_sampling, self.cover_stride = eval_stride, train_sampling, cover_stride\n        self.slices = int(slices)\n        self.slice_jitter = float(slice_jitter)\n        self.phys_mm = float(phys_mm)\n        assert self.slices % 2 == 1 and self.slices >= 1\n        index = json.load(open(os.path.join(px_dir, "index.json"), encoding="utf-8"))\n        # blocks carry (file, view_id, start): file 0 = the main cache, file 1 = the add-on cache (e.g. the v2 patellofemoral\n        # view built with px_cache --views v2add); a study missing from the add-on simply has no add-on windows\n        self.blocks = {u: [dict(b, views=[(0, v, st) for v, st in b["views"]]) for b in index["blocks"][u]] for u in uids}\n        self.dirs = [px_dir]\n        if px_extra:\n            extra = json.load(open(os.path.join(px_extra, "index.json"), encoding="utf-8"))\n            self.dirs.append(px_extra)\n            n_add = 0\n            for u in uids:\n                eb = extra["blocks"].get(u)\n                if not eb:\n                    continue\n                main = self.blocks[u]\n                if [b["series"] for b in eb] != [b["series"] for b in main]:\n                    raise SystemExit("add-on cache series order differs from the main cache for study " + u)\n                for b, e in zip(main, eb):\n                    b["views"] = b["views"] + [(1, v, st) for v, st in e["views"]]\n                    n_add += len(e["views"])\n            print("add-on cache %s: %d extra (series, view) blocks for %d studies" % (px_extra, n_add, len(uids)), flush=True)\n        if px_override:\n            # override cache (sidefix): the listed studies are read entirely from it (same format, own row space), e.g.\n            # the studies whose knee side the image check corrected, re-rendered through the inference path\n            if px_extra:\n                raise SystemExit("--px-override cannot be combined with --px-extra")\n            ov = json.load(open(os.path.join(px_override, "index.json"), encoding="utf-8"))\n            f_ov = len(self.dirs)\n            self.dirs.append(px_override)\n            n_ov = 0\n            for u in uids:\n                ob = ov["blocks"].get(u)\n                if ob is not None:\n                    self.blocks[u] = [dict(b, views=[(f_ov, v, st) for v, st in b["views"]]) for b in ob]\n                    n_ov += 1\n            print("override cache %s: %d of %d studies read from it" % (px_override, n_ov, len(uids)), flush=True)\n        self.px = self.meta = None\n        self.epoch = 0\n\n    def __len__(self):\n        return len(self.uids)\n\n    def __getitem__(self, i):\n        if self.px is None:\n            self.px = [np.load(os.path.join(d, "px.u8.npy"), mmap_mode="r") for d in self.dirs]\n            self.meta = [np.load(os.path.join(d, "meta.f32.npy"), mmap_mode="r") for d in self.dirs]\n        uid = self.uids[i]\n        rng = np.random.default_rng(self.seed + 1000003 * self.epoch + i) if self.mode == "train" else None\n        # \'cover\': every series/view is covered at stride cover_stride (x the physical step) with a random phase, so a\n        # focal finding spanning >= one stride is in the sampled set of every epoch (uniform sampling of ~10% of the\n        # windows misses a 3-slice finding in about half of the epochs).  Rows: (file, centre row, K slice rows k-r..k+r).\n        cand = window_rows(self.blocks[uid], lambda f, r: self.meta[f][r], self.mode, self.slices, self.eval_stride,\n                           self.cover_stride, self.phys_mm, rng, self.slice_jitter, self.train_sampling)\n        if self.mode == "train" and len(cand) > self.n_win:\n            cand = [cand[j] for j in np.sort(rng.choice(len(cand), self.n_win, replace=False))]\n        rows = np.array(cand, np.int64)\n        S = self.px[0].shape[-1]\n        img = np.empty((len(rows), self.slices, S, S), np.uint8)\n        m = np.empty((len(rows), self.meta[0].shape[1]), np.float32)\n        for f in range(len(self.dirs)):\n            sel = np.flatnonzero(rows[:, 0] == f)\n            if not len(sel):\n                continue\n            r = rows[sel]\n            img[sel] = np.stack([np.asarray(self.px[f][r[:, 2 + d]]) for d in range(self.slices)], 1)\n            m[sel] = np.asarray(self.meta[f][r[:, 1]])\n        meta = np.stack([m[:, 1], m[:, 3], m[:, 4], m[:, 5], m[:, 6], m[:, 7]], 1)   # slot, view, rel_pos, log_spacing, extent, fluid\n        if self.extent_meta:                                   # + series index, plane (0 sag / 1 cor / 2 ax), slices in the series\n            meta = np.concatenate([meta, np.stack([m[:, 0], m[:, 2], m[:, 10]], 1)], 1)\n        return torch.from_numpy(img), torch.from_numpy(meta.astype(np.float32)), i\n\n\ndef collate(batch):\n    W = max(len(b[0]) for b in batch)\n    S = batch[0][0].shape[-1]\n    img = torch.zeros((len(batch), W, batch[0][0].shape[1], S, S), dtype=torch.uint8)\n    meta = torch.zeros((len(batch), W, batch[0][1].shape[1]), dtype=torch.float32)\n    mask = torch.zeros((len(batch), W), dtype=torch.bool)\n    for j, (x, m, _) in enumerate(batch):\n        img[j, :len(x)] = x\n        meta[j, :len(x)] = m\n        mask[j, :len(x)] = True\n    return img, meta, mask, torch.tensor([b[2] for b in batch])\n\n\nOFL_BACKBONE = "ofl_l"\n\n\nclass OFLEncoder(nn.Module):\n    """OrthoFoundation-L (public self-supervised weights) as the window encoder, run exactly as the feature extractor\n    runs it: timm Eva vit_large_patch16_dinov3_qkvb, RoPE removed, 5 prefix tokens, ImageNet normalisation (done by\n    Net).  Feature = [CLS token, mean patch token] (2048).  The patch embedding, the tokens, norm_pre and the first\n    `freeze` of the 24 blocks are frozen and run without a graph; the remaining blocks and the final norm are trained.\n    `deploy_state_dict` returns only what differs from the public weights, so a fold model can be shipped small and\n    rebuilt on top of the public weights."""\n\n    def __init__(self, init, freeze=16, img_size=256):\n        super().__init__()\n        import timm\n        self.vit = timm.create_model("vit_large_patch16_dinov3_qkvb", pretrained=False, num_classes=0, global_pool="",\n                                     img_size=img_size)\n        if init:\n            if str(init).endswith(".safetensors"):\n                from safetensors.torch import load as load_bytes\n                with open(str(init), "rb") as fh:          # one sequential read; the mmap loader page-faults 4 KB at a time\n                    sd = load_bytes(fh.read())             # (0.4 MB/s from a USB disk: 48 min per fold on the 5090)\n            else:\n                sd = torch.load(init, map_location="cpu")\n            self.vit.load_state_dict(sd, strict=True)\n        self.vit.rope = None\n        self.freeze = int(freeze)\n        self.num_features = 2 * self.vit.embed_dim\n        self.frozen_keys = set()\n        frozen_modules = [self.vit.patch_embed, self.vit.norm_pre] + list(self.vit.blocks[: self.freeze])\n        for mod in frozen_modules:\n            for p in mod.parameters():\n                p.requires_grad_(False)\n        for name in ("cls_token", "reg_token", "pos_embed"):\n            t = getattr(self.vit, name, None)\n            if isinstance(t, nn.Parameter):\n                t.requires_grad_(False)\n        for k, p in self.vit.named_parameters():\n            if not p.requires_grad:\n                self.frozen_keys.add("vit." + k)\n\n    def trunk(self, x):\n        v = self.vit\n        with torch.no_grad():\n            x = v.patch_embed(x)\n            x, _ = v._pos_embed(x)\n            x = v.norm_pre(x)\n            for blk in v.blocks[: self.freeze]:\n                x = blk(x, rope=None)\n        return x\n\n    def tail(self, x):\n        v = self.vit\n        for blk in v.blocks[self.freeze:]:\n            x = blk(x, rope=None)\n        x = v.norm(x)\n        return torch.cat([x[:, 0], x[:, v.num_prefix_tokens:].mean(1)], -1)\n\n    def forward(self, x):\n        return self.tail(self.trunk(x))\n\n\nclass Net(nn.Module):\n    def __init__(self, backbone, init, hidden=384, drop=0.1, freeze_blocks=16, crop=256, state_spec=None, in_chans=3, spatial=False,\n                 extent=False):\n        super().__init__()\n        import timm\n        self.backbone = backbone\n        self.crop = int(crop)\n        self.in_chans = int(in_chans)\n        self.spatial = bool(spatial)\n        self.extent = bool(extent)\n        self._ext_area = None\n        if self.spatial and backbone == OFL_BACKBONE:\n            raise SystemExit("--spatial-attn needs a timm backbone (the OFL encoder returns pooled features)")\n        if backbone == OFL_BACKBONE:\n            assert self.in_chans == 3, "the OFL patch embedding is a public 3-channel weight"\n            self.enc = OFLEncoder(init, freeze=freeze_blocks, img_size=crop)\n        else:\n            self.enc = self._timm_encoder(backbone, init, self.in_chans)\n        self.has_bn = any(isinstance(m, nn.modules.batchnorm._BatchNorm) for m in self.enc.modules())\n        self._build_head(hidden, drop)\n        self.state_spec = state_spec\n        self.has_state = state_spec is not None\n        if self.has_state:\n            self._build_state_head(hidden, state_spec)\n        if self.spatial:                        # created last: every baseline parameter keeps the baseline\'s initial values\n            d = self.enc.num_features\n            self.sp_key = nn.Linear(d, 128)     # keys from per-position normalised features\n            self.sp_pos = nn.Linear(16, 128, bias=False)\n            nn.init.zeros_(self.sp_pos.weight)\n            self.sp_q = nn.Parameter(torch.zeros(C, 128))   # zero query = uniform weights = the baseline average\n        if self.extent:                         # OA defect extent (after every other parameter: same init as the baseline)\n            self.ext_u = nn.Parameter(torch.zeros(C, hidden))\n            self.ext_b = nn.Parameter(torch.full((C,), -4.0))   # sigmoid(-4) = 0.018: area starts ~4 mm^2, not ~3,000 (audit 3)\n            self.ext_gamma = nn.Parameter(torch.zeros(C))    # zero = the baseline logit at init\n            self.ext_view = nn.Parameter(torch.zeros(C, 6))  # per (class, view) weight inside a series; 0 = uniform (audit 4)\n            em = torch.zeros(C)\n            for l in ("Medial OA", "Lateral OA", "PF OA"):\n                em[LABELS.index(l)] = 1.0\n            self.register_buffer("ext_mask", em)\n\n    def _build_state_head(self, hidden, spec):\n        """Report-state heads: for class c a K_c-way read-out of the class token (which report state the images show),\n        OA-sign / acuity binaries on the parent class token, and the frozen prior vector per state (mean report prior of the state\'s category over the pool, not fitted on gold).\n        The class logit gains  gate_c * (logit(sum_s p(s|x) prior_c(s)) - logit(base_c)): the \'align to the official\n        label\' mapping applied to predicted states; gate starts at 0 so the direct path is untouched at init."""\n        cats = spec["categories"]\n        self.state_k = [len(cats[l]) for l in LABELS]\n        kmax = max(self.state_k)\n        self.state_w = nn.Parameter(torch.randn(C, kmax, hidden) * .02)\n        self.state_b = nn.Parameter(torch.zeros(C, kmax))\n        prior = torch.full((C, kmax), float("nan"))\n        valid = torch.zeros((C, kmax), dtype=torch.bool)\n        for i, l in enumerate(LABELS):\n            for j, pv in enumerate(spec["prior"][l]):\n                prior[i, j] = float(pv); valid[i, j] = True\n        self.register_buffer("state_prior", torch.nan_to_num(prior, nan=0.5).clamp(1e-3, 1 - 1e-3))\n        self.register_buffer("state_valid", valid)\n        self.register_buffer("state_base_logit", torch.logit(torch.tensor([float(spec["base"][l]) for l in LABELS]).clamp(1e-3, 1 - 1e-3)))\n        self.state_gate = nn.Parameter(torch.zeros(C))\n        self.bin_names = [(b, l) for b, labs in spec["binaries"].items() for l in labs]\n        self.bin_parent = [LABELS.index(l) for _, l in self.bin_names]\n        self.bin_w = nn.Parameter(torch.randn(len(self.bin_names), hidden) * .02)\n        self.bin_b = nn.Parameter(torch.zeros(len(self.bin_names)))\n\n    def state_logits(self, study):\n        """study: [B, C, hidden] class tokens -> state logits [B, C, Kmax] (invalid slots -inf), binary logits [B, nbin]."""\n        sl = torch.einsum("bch,ckh->bck", study, self.state_w).float() + self.state_b\n        sl = sl.masked_fill(~self.state_valid[None], float("-inf"))\n        bl = torch.einsum("bjh,jh->bj", study[:, self.bin_parent], self.bin_w).float() + self.bin_b\n        return sl, bl\n\n    def state_evidence(self, sl):\n        """logit of the official probability implied by the predicted state distribution, relative to the class base rate."""\n        p = sl.softmax(-1)                                          # -inf slots -> 0\n        e = (p * self.state_prior[None]).sum(-1).clamp(1e-3, 1 - 1e-3)\n        return torch.logit(e) - self.state_base_logit[None]\n\n    def deploy_state_dict(self):\n        """Everything except the frozen public OFL weights (identical to state_dict() for other backbones)."""\n        frozen = {"enc." + k for k in getattr(self.enc, "frozen_keys", set())}\n        return {k: v.detach().cpu() for k, v in self.state_dict().items() if k not in frozen}\n\n    def load_deploy_state_dict(self, sd):\n        missing, unexpected = self.load_state_dict(sd, strict=False)\n        frozen = {"enc." + k for k in getattr(self.enc, "frozen_keys", set())}\n        if unexpected or set(missing) - frozen:\n            raise ValueError("deploy state does not match: missing %s unexpected %s" % (sorted(set(missing) - frozen)[:5], unexpected[:5]))\n\n    def _timm_encoder(self, backbone, init, in_chans=3):\n        import timm\n        # init == "timm": public ImageNet weights from the local hub cache (HF_HOME); a path: a state dict file\n        is_vit = backbone.startswith("vit_")\n        kw = dict(pretrained=(init == "timm"), num_classes=0, in_chans=in_chans,\n                  drop_path_rate=0.1 if ("convnext" in backbone or is_vit) else 0.0)\n        if is_vit:\n            kw["dynamic_img_size"] = True       # keeps the pretrained grid (DINOv2 518 / DINOv3 256); position embeddings or RoPE\n        else:                                   # are interpolated to the crop at forward; pooling = the model\'s own (token / avg)\n            kw["global_pool"] = "avg"\n        if backbone.startswith(("coatnet", "maxvit", "maxxvit")):\n            kw["img_size"] = self.crop          # relative-position tables are built for this input size (CoAtNet / MaxViT)\n        if self.spatial:                        # keep the spatial grid: CNN -> normed map [N, d, h, w], ViT -> all tokens [N, T, d]\n            kw["global_pool"] = ""\n        enc = timm.create_model(backbone, **kw)\n        if init and init != "timm":\n            if str(init).endswith(".safetensors"):\n                from safetensors.torch import load as load_bytes\n                with open(str(init), "rb") as fh:\n                    sd = load_bytes(fh.read())\n            else:\n                sd = torch.load(init, map_location="cpu")\n                sd = sd.get("state_dict", sd.get("model", sd)) if isinstance(sd, dict) else sd\n            if in_chans != 3:                       # inflate the pretrained RGB stem: mean over RGB, replicated K times, scaled 3/K\n                for k, w in list(sd.items()):\n                    if w.ndim == 4 and w.shape[1] == 3 and k in enc.state_dict() and enc.state_dict()[k].shape[1] == in_chans:\n                        sd[k] = w.mean(1, keepdim=True).repeat(1, in_chans, 1, 1) * (3.0 / in_chans)\n                        print("inflated %s %s -> %s" % (k, tuple(w.shape), tuple(sd[k].shape)), flush=True)\n            missing, unexpected = enc.load_state_dict(sd, strict=False)\n            print("init %s: missing %d unexpected %d" % (init, len(missing), len(unexpected)), flush=True)\n        self.sp_head_norm = None\n        if self.spatial and isinstance(getattr(getattr(enc, "head", None), "norm", None), nn.Module) \\\n                and not isinstance(enc.head.norm, nn.Identity):\n            # CNN heads pool THEN normalise: keep that order per class (pool the raw map, then this norm on each class\'s\n            # pooled vector), so a zero spatial query reproduces the baseline feature LN(mean_p x_p) exactly\n            self.sp_head_norm = enc.head.norm\n            enc.head.norm = nn.Identity()\n        return enc\n\n    def _build_head(self, hidden, drop):\n        d = self.enc.num_features\n        self.proj = nn.Sequential(nn.Linear(d, hidden), nn.LayerNorm(hidden))\n        self.slot = nn.Embedding(10, hidden)\n        self.view = nn.Embedding(6, hidden)\n        self.geom = nn.Linear(4, hidden, bias=False)\n        for m in (self.slot, self.view):\n            nn.init.zeros_(m.weight)\n        nn.init.zeros_(self.geom.weight)\n        self.drop = nn.Dropout(drop)\n        self.win_proj = nn.Linear(hidden, hidden)\n        self.q_win = nn.Parameter(torch.randn(C, hidden) * .02)\n        self.out_norm = nn.LayerNorm(hidden)\n        self.cls_w = nn.Parameter(torch.randn(C, hidden) * .02)\n        self.bias = nn.Parameter(torch.zeros(C))\n        if self.in_chans == 3:\n            self.register_buffer("mean", torch.tensor([.485, .456, .406]).view(1, 3, 1, 1))\n            self.register_buffer("std", torch.tensor([.229, .224, .225]).view(1, 3, 1, 1))\n        else:                                       # grey slices: one ImageNet-mean/std per channel\n            self.register_buffer("mean", torch.full((1, self.in_chans, 1, 1), .449))\n            self.register_buffer("std", torch.full((1, self.in_chans, 1, 1), .226))\n\n    def spatial_pool(self, x):\n        """Encoder output with its spatial grid ([N, d, h, w] CNN map or [N, T, d] ViT tokens) -> per-class window features\n        [N, C, d]: class c weighs the window\'s positions with its own query (sp_q[c]; zero at init = the plain average)\n        over keys tanh(W f(p) + W_pos pe(p)), so a class can read its own structure inside a window shared with others."""\n        if x.ndim == 4:\n            h, w = x.shape[-2:]\n            f = x.flatten(2).transpose(1, 2)                                      # [N, h*w, d]\n        else:\n            f = x[:, getattr(self.enc, "num_prefix_tokens", 0):]\n            h = w = int(round(f.shape[1] ** 0.5))\n            assert h * w == f.shape[1], "non-square ViT token grid"\n        yy, xx = torch.meshgrid(torch.linspace(-1, 1, h, device=f.device), torch.linspace(-1, 1, w, device=f.device), indexing="ij")\n        fr = torch.tensor([1., 2., 4., 8.], device=f.device) * (math.pi / 2)\n        yy, xx = yy.reshape(-1, 1) * fr, xx.reshape(-1, 1) * fr\n        pe = torch.cat([yy.sin(), yy.cos(), xx.sin(), xx.cos()], 1)              # [h*w, 16] fixed 2-D position features\n        kin = F.layer_norm(f.float(), (f.shape[-1],)).to(f.dtype)                # keys: per-position normalised features\n        k = torch.tanh(self.sp_key(kin) + self.sp_pos(pe.to(f.dtype))[None])    # [N, P, 128]\n        att = torch.einsum("npk,ck->npc", k, self.sp_q.to(k.dtype)).float().softmax(1).to(f.dtype)\n        g = torch.einsum("npc,npd->ncd", att, f)                                 # [N, C, d]\n        if self.sp_head_norm is not None:                                        # CNN: the head norm after pooling, per class\n            n_ = self.sp_head_norm\n            g = F.layer_norm(g.float(), (g.shape[-1],), n_.weight, n_.bias, n_.eps).to(f.dtype)\n        return g\n\n    def extent_area(self, e, meta, mask):\n        """e [B, W, C] window defect logits -> A [B, C] (mm^2, an area proxy): inside each series the mean sigmoid(e) of\n        each view\'s windows, combined over the views present by a learned per-(class, view) softmax (uniform at init: a\n        compartment sits inside only some of the locals), times the series\' slab thickness (slices x spacing) = mm of\n        slab with defect evidence; per plane the max over series; then sagittal x coronal (tibiofemoral OA) or axial x\n        sagittal (PF OA); other classes 0."""\n        V = 6\n        sid = meta[..., 6].long()\n        # one slot per series, never merged: 16 covers every training study (at most 14 series); a deployed study with more\n        # series gets more slots (lossless -- an empty slot has mean 0 and the per-plane max runs over values >= 0)\n        S_MAX = max(16, int(sid.max()) + 1)\n        vid = meta[..., 1].long().clamp(0, V - 1)\n        q = torch.sigmoid(e.float()) * mask[..., None]\n        ohv = F.one_hot(sid * V + vid, S_MAX * V).float() * mask[..., None]     # [B, W, S*V]\n        cntv = ohv.sum(1)                                                          # [B, S*V]\n        mean_v = (torch.einsum("bwk,bwc->bkc", ohv, q) / cntv.clamp_min(1)[..., None]).view(len(e), S_MAX, V, -1)\n        present = (cntv > 0).view(len(e), S_MAX, V, 1)\n        wv = self.ext_view.float().t()[None, None].masked_fill(~present, -1e4).softmax(2) * present   # [B, S, V, C]\n        mean = (wv * mean_v).sum(2)                                                # [B, S, C]; 0 for a series without windows\n        oh = F.one_hot(sid, S_MAX).float() * mask[..., None]                     # [B, W, S]\n        cnt = oh.sum(1)                                                            # [B, S]\n        span = torch.expm1(meta[..., 3].float()).clamp_min(0) * meta[..., 8].float()   # slab thickness (mm) of each window\'s series\n        span_s = torch.einsum("bws,bw->bs", oh, span) / cnt.clamp_min(1)\n        plane_s = (torch.einsum("bws,bw->bs", oh, meta[..., 7].float()) / cnt.clamp_min(1)).round()\n        E = mean * span_s[..., None]                                               # [B, S, C] mm of slab with defect evidence\n        per_plane = []\n        for pl in (0, 1, 2):\n            ok = ((plane_s == pl) & (cnt > 0))[..., None]\n            per_plane.append(E.masked_fill(~ok, 0.0).amax(1))                     # [B, C]; 0 when the plane is absent\n        sag, cor, ax = per_plane\n        area = torch.zeros_like(sag)\n        for l in ("Medial OA", "Lateral OA"):\n            c = LABELS.index(l)\n            area[:, c] = sag[:, c] * cor[:, c]\n        c = LABELS.index("PF OA")\n        area[:, c] = ax[:, c] * sag[:, c]\n        return area\n\n    def head(self, tok, meta, mask, return_aux=False):\n        """Encoder output of the real windows [N, d] (or [N, C, d] with --spatial-attn) -> class logits [B, C] (and the\n        state/binary logits when asked)."""\n        B, W = mask.shape\n        f = self.proj(tok.float())\n        m = meta\n        if self.spatial:                                                          # class-specific window tokens [B, W, C, H]\n            emb = self.slot(m[..., 0].long().clamp(0, 9)) + self.view(m[..., 1].long().clamp(0, 5)) + self.geom(m[..., 2:6])\n            t = f.new_zeros((B, W) + tuple(f.shape[1:]))\n            t[mask] = f\n            t = self.drop(t + emb[:, :, None, :])\n            s2 = torch.einsum("bwch,ch->bwc", torch.tanh(self.win_proj(t)), self.q_win).float()\n            s2 = s2.masked_fill(~mask[:, :, None], float("-inf"))\n            a2 = s2.softmax(1).to(t.dtype)\n            study = self.out_norm(torch.einsum("bwc,bwch->bch", a2, t))\n            if self.extent:\n                e_win = (t.float() * self.ext_u[None, None]).sum(-1) + self.ext_b\n        else:\n            t = f.new_zeros((B, W, f.shape[-1]))\n            t[mask] = f\n            t = t + self.slot(m[..., 0].long().clamp(0, 9)) + self.view(m[..., 1].long().clamp(0, 5)) + self.geom(m[..., 2:6])\n            t = self.drop(t)\n            s2 = torch.einsum("bwh,ch->bwc", torch.tanh(self.win_proj(t)), self.q_win).float()\n            s2 = s2.masked_fill(~mask[:, :, None], float("-inf"))\n            a2 = s2.softmax(1).to(t.dtype)\n            study = self.out_norm(torch.einsum("bwc,bwh->bch", a2, t))\n            if self.extent:\n                e_win = torch.einsum("bwh,ch->bwc", t.float(), self.ext_u) + self.ext_b\n        z = (study * self.cls_w).sum(-1).float() + self.bias\n        if self.extent:\n            area = self.extent_area(e_win, meta, mask)\n            self._ext_area = area\n            z = z + self.ext_gamma * self.ext_mask * torch.log1p(area / 100.0)\n        if not self.has_state:\n            return (z, None, None) if return_aux else z\n        sl, bl = self.state_logits(study)\n        z = z + self.state_gate[None] * self.state_evidence(sl)\n        return (z, sl, bl) if return_aux else z\n\n    def forward(self, img, meta, mask, chunk=0, return_aux=False):\n        x = img[mask]                                               # only real windows go through the encoder\n        n = len(x)\n        if n % 32 and (not self.training or not self.has_bn):     # pad the encoder batch to a multiple of 32: a handful of shapes\n            x = torch.cat([x, x.new_zeros((32 - n % 32,) + tuple(x.shape[1:]))])   # instead of one per step (allocator, cuDNN autotune);\n        feats = []\n        step = chunk or len(x)\n        for s in range(0, len(x), step):\n            xs = x[s:s + step].float().div_(255.)\n            xs = ((xs - self.mean) / self.std).contiguous(memory_format=torch.channels_last)\n            e = self.enc(xs)\n            feats.append(self.spatial_pool(e) if self.spatial else e)\n        return self.head(torch.cat(feats)[:n], meta, mask, return_aux=return_aux)\n\n\ndef bce_soft(z, target, weight):\n    loss = F.binary_cross_entropy_with_logits(z, target, reduction="none") * weight\n    den = weight.sum(0)\n    ok = den > 0\n    return (loss.sum(0)[ok] / den[ok]).mean()\n\n\ndef lr_loss(z, lr):\n    return (-torch.logaddexp(F.logsigmoid(z) + torch.log(lr), F.logsigmoid(-z))).mean(0).mean()\n\n\ndef state_loss(sl, bl, st_idx, st_w, bin_t):\n    """Masked cross-entropy over report states (per class, weight-normalised) + masked BCE over the binaries.\n    st_idx [B, C] long (-1 unknown), st_w [B, C], bin_t [B, nbin] (nan unknown)."""\n    total, n = sl.new_zeros(()), 0\n    for c in range(sl.shape[1]):\n        ok = st_idx[:, c] >= 0\n        if ok.any():\n            ce = F.cross_entropy(sl[ok, c], st_idx[ok, c], reduction="none") * st_w[ok, c]\n            total = total + ce.sum() / st_w[ok, c].sum().clamp_min(1e-6); n += 1\n    known = ~torch.isnan(bin_t)\n    if known.any():\n        t = torch.nan_to_num(bin_t)\n        bce = F.binary_cross_entropy_with_logits(bl, t, reduction="none") * known\n        per = bce.sum(0) / known.sum(0).clamp_min(1)\n        total = total + per[known.any(0)].mean(); n += 1\n    return total / max(n, 1)\n\n\nclass PairBatchSampler:\n    """Training batches of `batch` dataset indices.  With probability `frac` a batch opens with a matched pair -- a positive\n    and a hard negative of one pair-family class that share the co-finding signature (bucket drawn with the given\n    probabilities) -- and is filled with base draws; otherwise all draws are base draws (weighted with replacement, as\n    the rare-boost sampler).  len = n // batch: the same number of steps per epoch as the plain loader."""\n\n    def __init__(self, n, batch, weights, buckets, frac, seed):\n        self.n, self.batch, self.frac = int(n), int(batch), float(frac)\n        self.w = torch.ones(self.n, dtype=torch.double) if weights is None else torch.as_tensor(weights, dtype=torch.double)\n        self.buckets = buckets\n        self.bprob = torch.tensor([b[2] for b in buckets], dtype=torch.double) if buckets else None\n        self.generator = torch.Generator().manual_seed(int(seed))\n        assert self.batch >= 2\n\n    def __len__(self):\n        return self.n // self.batch\n\n    def _base(self, k):\n        return torch.multinomial(self.w, k, replacement=True, generator=self.generator).tolist() if k > 0 else []\n\n    def __iter__(self):\n        g = self.generator\n        for _ in range(len(self)):\n            if self.buckets and float(torch.rand(1, generator=g)) < self.frac:\n                pos, neg, _ = self.buckets[int(torch.multinomial(self.bprob, 1, generator=g))]\n                i = pos[int(torch.randint(0, len(pos), (1,), generator=g))]\n                j = neg[int(torch.randint(0, len(neg), (1,), generator=g))]\n                yield [int(i), int(j)] + self._base(self.batch - 2)\n            else:\n                yield self._base(self.batch)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--px", required=True)\n    ap.add_argument("--px-extra", default=None, help="add-on pixel cache (px_cache --views v2add) read next to --px; its windows join every study")\n    ap.add_argument("--px-override", default=None, help="override cache (sidefix): studies listed there are read from it instead of --px")\n    ap.add_argument("--targets", required=True)\n    ap.add_argument("--extra", default=None)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--arms", default="E_prior")\n    ap.add_argument("--backbone", default="resnet34")\n    ap.add_argument("--init", default="F:/rsnaknee/pretrained/resnet34-imagenet.pth")\n    ap.add_argument("--seeds", type=int, default=1)\n    ap.add_argument("--folds", type=int, default=5)\n    ap.add_argument("--epochs", type=int, default=10)\n    ap.add_argument("--batch", type=int, default=8)\n    ap.add_argument("--train-windows", type=int, default=48)\n    ap.add_argument("--crop", type=int, default=256)\n    ap.add_argument("--slices", type=int, default=3, help="adjacent slices per window = input channels (odd; CNN backbones only beyond 3)")\n    ap.add_argument("--slice-jitter", type=float, default=0.0, help="probability that a training window uses neighbours two slices apart")\n    ap.add_argument("--ema", type=float, default=0.0, help="EMA decay of the trainable weights (0 = off); EMA weights are evaluated and saved")\n    ap.add_argument("--bn-recal-batches", type=int, default=0, help="v7: BatchNorm backbones only (CoAtNet, ResNet): after --ema copies the "\n                    "averaged weights in, recompute the BatchNorm running statistics for them over this many training batches "\n                    "(centre crop, no augmentation) before evaluating and saving; also applied by --reeval-run.  0 = off "\n                    "(backbones without BatchNorm are never touched)")\n    ap.add_argument("--include-review-holdout", action="store_true", help="add the 120 review-holdout studies (full report targets, "\n                    "fold -3) to the training pool; they were held out for the report-reader evaluation, not for image training")\n    ap.add_argument("--grad-ckpt", action="store_true", help="gradient checkpointing in the timm encoder (keeps 208 windows/study on 32 GB cards)")\n    ap.add_argument("--phys-mm", type=float, default=0.0, help="window spacing in mm of anatomy: the covering stride and the K-slice "\n                    "neighbour step are multiplied by round(phys_mm / slice spacing), so a 0.4 mm 3-D stack is sampled like a 3.5 mm "\n                    "2-D series (0 = per-slice, the old behaviour)")\n    ap.add_argument("--rare-boost", action="store_true", help="oversample studies with rare report states (MCL>=2/acute, fracture, high PF OA, Baker>=moderate, contusion); needs --state-targets")\n    ap.add_argument("--lr", type=float, default=3e-4)\n    ap.add_argument("--freeze-blocks", type=int, default=16, help="ofl_l backbone: number of leading transformer blocks kept frozen (of 24)")\n    ap.add_argument("--llrd", type=float, default=1.0, help="ViT backbones (ofl_l, timm vit_*): layer-wise learning-rate decay per block from the top (1 = off); block i of n gets lr * llrd**(n - 1 - i), embeddings lr * llrd**n")\n    ap.add_argument("--head-lr", type=float, default=1e-3)\n    ap.add_argument("--wd", type=float, default=0.02)\n    ap.add_argument("--workers", type=int, default=6)\n    ap.add_argument("--eval-workers", type=int, default=3)\n    ap.add_argument("--forward-only", action="store_true", help="one batch forward/backward, NO optimizer step, then exit")\n    ap.add_argument("--compile", action="store_true", help="torch.compile the encoder in place (dynamic shapes); same maths, 1.4x (ConvNeXt-small) / 1.9x (DINOv3-B) on a PRO 6000; weight names unchanged")\n    ap.add_argument("--resume", action="store_true", help="continue every fold from its end-of-epoch checkpoint in <out>/ckpt when present")\n    ap.add_argument("--ckpt-every", type=int, default=1, help="write the fold checkpoint every N epochs (0 = never); removed when the fold ends")\n    ap.add_argument("--stop-after-epoch", type=int, default=-1, help="testing: exit right after the checkpoint of this epoch")\n    ap.add_argument("--eval-stride", type=int, default=3, help="evaluation windows: every k-th slice as centre (3 = the validated default; 1 = every slice)")\n    ap.add_argument("--train-sampling", default="uniform", choices=["uniform", "cover"],\n                    help="uniform = --train-windows random windows of the study; cover = every series/view at --cover-stride with a random phase (capped at --train-windows)")\n    ap.add_argument("--cover-stride", type=int, default=3)\n    ap.add_argument("--state-targets", default=None, help="state_targets_<tag>.csv from make_state_targets.py (report-state construct)")\n    ap.add_argument("--state-spec", default=None, help="state_spec_<tag>.json (categories, prior vectors, binaries)")\n    ap.add_argument("--state-weight", type=float, default=0.5, help="weight of the state/binary losses next to the arm loss")\n    ap.add_argument("--spatial-attn", action="store_true", help="per-class spatial attention inside each window (encoder keeps its "\n                    "grid; class queries start at zero = plain average), then per-class window attention over class-specific tokens")\n    ap.add_argument("--pair-rank", type=float, default=0.0, help="weight of the matched-pair ranking loss on the 8 pair families "\n                    "(positive vs hard negative of a class with the same co-finding signature); needs --state-targets; 0 = off")\n    ap.add_argument("--pair-frac", type=float, default=0.33, help="share of training batches that open with a matched pair (--pair-rank)")\n    ap.add_argument("--seed-base", type=int, default=0, help="first seed (seeds seed_base .. seed_base+seeds-1); a replicate of a recipe")\n    ap.add_argument("--extent", action="store_true", help="M4: OA defect extent per series and plane (area proxy) into the OA logits")\n    ap.add_argument("--extent-weight", type=float, default=0.2, help="weight of the extent supervision (>= 1 cm^2 stated vs normal/low grade)")\n    ap.add_argument("--holdout-fold", type=int, default=None,\n                    help="clean ruler fold for pseudo-label rounds: this hash fold is excluded from EVERY training set (no model is trained "\n                         "for it); its predictions are the mean of the other folds\' models, so a teacher->student chain evaluated on it "\n                         "with local_ruler --folds-only is leak-free")\n    ap.add_argument("--reeval-run", default=None, help="predict-only: load models/<arm>_s<seed>_f<k>.pt from this run directory and rewrite the out-of-fold "\n                    "predictions into --out with the current --eval-stride (no training)")\n    ap.add_argument("--aug", default="crop", choices=["crop", "v2"], help="v2 = crop + per-study rotation/scale + gamma/contrast jitter (no flips: laterality)")\n    ap.add_argument("--aug-res", type=float, default=0.0, help="v2 only: probability that a training study is down-sampled in-plane by "\n                    "1.25-2x and up-sampled back (simulates 0.33-0.5 mm scanners); 0 = off, random stream unchanged")\n    ap.add_argument("--only-folds", default=None, help="diagnostic: comma list of folds to train (others skipped); outputs are written with a partial_ prefix and must not be used as OOF")\n    ap.add_argument("--check-targets", action="store_true", help="pre-launch check on CPU: build the pool and every arm\'s targets and the "\n                    "state targets with the training code path, print a summary, and exit (no data loading, no model, no GPU)")\n    ap.add_argument("--i-have-user-authorisation-to-finetune", action="store_true")\n    a = ap.parse_args()\n    if not a.forward_only and not a.check_targets and not a.i_have_user_authorisation_to_finetune:\n        raise SystemExit("refusing to update an encoder: the user has only authorised light heads on cached features. "\n                         "Use --forward-only, or pass the authorisation flag after the user says so.")\n    os.makedirs(a.out, exist_ok=True)\n    dev = torch.device("cpu") if a.check_targets else torch.device("cuda")\n    torch.backends.cudnn.benchmark = True\n    index = json.load(open(os.path.join(a.px, "index.json"), encoding="utf-8"))\n    done = {u for u, v in index["done"].items() if v[1] > 0}\n    rows = list(csv.DictReader(open(a.targets, encoding="utf-8")))\n    by_uid = {r["StudyInstanceUID"]: r for r in rows}\n    if a.extra:\n        for r in csv.DictReader(open(a.extra, encoding="utf-8")):\n            if r["StudyInstanceUID"] in by_uid:\n                by_uid[r["StudyInstanceUID"]].update({k: v for k, v in r.items() if k != "StudyInstanceUID"})\n    gold = [r["StudyInstanceUID"] for r in rows if r["is_gold"] == "1" and r["StudyInstanceUID"] in done]\n    # a non-gold study whose report labels were blanked (e.g. the report describes another body part) carries no target:\n    # it leaves the pool, exactly like a gold row (logged below; bad_report_studies.json lists the blanked studies)\n    blanked = {r["StudyInstanceUID"] for r in rows if r["is_gold"] == "0" and any(r.get(L + "|prior", "") == "" for L in LABELS)}\n    pool = sorted(r["StudyInstanceUID"] for r in rows if r["is_gold"] == "0" and r["StudyInstanceUID"] in done\n                  and r["StudyInstanceUID"] not in blanked\n                  and ((r["review_holdout"] == "0" and int(r["fold"]) != -3) or a.include_review_holdout))\n    assert not (set(pool) & set(gold))\n    for u in gold:\n        assert all(by_uid[u][L + "|strict"] == "" and by_uid[u][L + "|prior"] == "" for L in LABELS)\n    K = a.folds\n    pool_fold = np.array([int(hashlib.sha1(u.encode()).hexdigest()[:8], 16) % K for u in pool])\n    print("pool %d gold %d folds %s | non-gold studies with blanked report labels left out: %d" % (\n        len(pool), len(gold), np.bincount(pool_fold, minlength=K).tolist(), len(blanked & done)), flush=True)\n\n    def col(uids, suffix, default):\n        out = np.full((len(uids), C), default, np.float32)\n        for i, u in enumerate(uids):\n            for j, L in enumerate(LABELS):\n                v = by_uid[u].get(L + "|" + suffix, "")\n                if v != "":\n                    out[i, j] = float(v)\n        return torch.from_numpy(out).to(dev)\n\n    specs = {}\n    for item in a.arms.split(","):\n        if "=" in item:\n            name, spec = item.split("=", 1)\n            specs[name] = tuple(spec.split(":"))\n\n    def arm_tensors(arm):\n        if arm == "S_strict":\n            return "bce", col(pool, "strict", 0.), col(pool, "strict_w", 0.)\n        if arm == "P_presence":\n            return "bce", col(pool, "presence", 0.), col(pool, "presence_w", 0.)\n        spec = {"E_prior": ("bce", "prior"), "E_lrprior": ("lr", "lr_prior")}.get(arm) or specs[arm]\n        kind, column = spec[0], spec[1]\n        if len(spec) > 2 and spec[2]:                       # name=kind:column:weight_column  (0 = cell carries no information)\n            w = col(pool, spec[2], 0.)\n            t = col(pool, column, 0.)\n            return kind, t, w\n        t = col(pool, column, float("nan"))\n        assert not torch.isnan(t).any()\n        return kind, t, torch.ones_like(t)\n\n    bias0 = torch.logit(col(pool, "prior", float("nan")).mean(0).clamp(1e-3, 1 - 1e-3))\n\n    state_spec = None\n    if a.state_targets:\n        state_spec = json.load(open(a.state_spec, encoding="utf-8"))\n        srows = {r["StudyInstanceUID"]: r for r in csv.DictReader(open(a.state_targets, encoding="utf-8"))}\n        st_idx = torch.full((len(pool), C), -1, dtype=torch.long)\n        st_w = torch.zeros((len(pool), C))\n        bin_names = [(b, l) for b, labs in state_spec["binaries"].items() for l in labs]\n        bin_t = torch.full((len(pool), len(bin_names)), float("nan"))\n        for i, u in enumerate(pool):\n            r = srows.get(u)\n            if r is None:\n                continue\n            for c, l in enumerate(LABELS):\n                v = r.get(l + "|state", "")\n                if v != "":\n                    st_idx[i, c] = int(v); st_w[i, c] = float(r.get(l + "|sw") or 1.0)\n            for j, (b, l) in enumerate(bin_names):\n                v = r.get(l + "|" + b, "")\n                if v != "":\n                    bin_t[i, j] = float(v)\n        st_idx, st_w, bin_t = st_idx.to(dev), st_w.to(dev), bin_t.to(dev)\n        print("state targets: known cells per class %s | binaries known %s" % (\n            (st_idx >= 0).sum(0).tolist(), (~torch.isnan(bin_t)).sum(0).tolist()), flush=True)\n\n    # ---- P1 pair families: class, positive states, hard-negative states, co-finding variables (from the report states)\n    pair_fam, pair_info = [], {}\n    if a.pair_rank > 0:\n        if state_spec is None:\n            raise SystemExit("--pair-rank needs --state-targets")\n        cats = state_spec["categories"]\n        Li = {l: c for c, l in enumerate(LABELS)}\n        bni = {(b, l): j for j, (b, l) in enumerate(bin_names)}\n        sti, bint = st_idx.cpu().numpy(), bin_t.cpu().numpy()\n\n        def sset(l, *names):\n            return [cats[l].index(x) for x in names]\n\n        def inset(l, idx_):                     # 1 / 0 per study, -1 when the report is silent on l\n            v = sti[:, Li[l]]\n            return np.where(v < 0, -1, np.isin(v, idx_).astype(np.int64))\n\n        def osteo(l):                           # "osteophytes written or not" (the table has 0 only for a normal compartment)\n            v = bint[:, bni[("sign_osteophytes", l)]]\n            return (v == 1).astype(np.int64)\n\n        TORN_M, HIGH = ("tear", "macerated_or_destroyed"), ("high_grade_size_unstated", "high_grade_large")\n        acl_t = inset("ACL", sset("ACL", "partial_tear_unspecified", "tear_unspecified", "complete_tear"))\n        mm_t = inset("Medial Meniscus", sset("Medial Meniscus", *TORN_M))\n        lm_t = inset("Lateral Meniscus", sset("Lateral Meniscus", *TORN_M))\n        hi = {l: inset(l, sset(l, *HIGH)) for l in ("Medial OA", "Lateral OA", "PF OA")}\n        cont = inset("Contusion", sset("Contusion", "contusion_or_bruise", "traumatic_marrow_edema"))\n        ev = sti[:, Li["Effusion"]]\n        eff_b = np.select([ev < 0, ev == 0, ev == 1, np.isin(ev, [2, 3])], [-1, 0, 1, 2], 3)\n        fams = [("Lateral Meniscus", TORN_M, ("normal", "degeneration_no_tear"), [acl_t, mm_t, hi["Lateral OA"]]),\n                ("Medial Meniscus", TORN_M, ("normal", "degeneration_no_tear"), [acl_t, lm_t, hi["Medial OA"]]),\n                ("Lateral OA", HIGH, ("normal", "low_grade_chondral"), [osteo("Lateral OA"), hi["Medial OA"], hi["PF OA"]]),\n                ("Medial OA", HIGH, ("normal", "low_grade_chondral"), [osteo("Medial OA"), hi["Lateral OA"], hi["PF OA"]]),\n                ("PF OA", HIGH, ("normal", "low_grade_chondral"), [osteo("PF OA"), hi["Medial OA"], hi["Lateral OA"]]),\n                ("Synovitis", ("synovitis", "synovial_thickening"), ("other_synovial_finding", "none"), [eff_b]),\n                ("MCL", ("partial_tear_or_grade2", "high_grade_partial_tear", "complete_tear"),\n                 ("normal", "periligamentous_edema_or_grade1", "chronic_or_healed_change"), [mm_t, cont, acl_t]),\n                ("ACL", ("partial_tear_unspecified", "tear_unspecified", "complete_tear"),\n                 ("normal", "signal_or_degeneration", "low_grade_sprain_or_partial"), [cont, lm_t, mm_t])]\n        role = np.zeros((len(pool), C), np.int64)            # +1 positive / -1 hard negative of the class\'s family / 0\n        sig = np.zeros((len(pool), C), np.int64)             # co-finding signature: pairs only within one signature\n        pw = np.zeros((len(pool), C), np.float32)             # report prior of the study\'s state (pair weight)\n        for l, pos_s, neg_s, cof in fams:\n            c = Li[l]\n            v = sti[:, c]\n            role[np.isin(v, sset(l, *pos_s)), c] = 1\n            role[np.isin(v, sset(l, *neg_s)), c] = -1\n            unknown_cof = np.zeros(len(pool), bool)\n            for arr in cof:\n                unknown_cof |= np.asarray(arr) < 0\n            role[unknown_cof, c] = 0                 # an unknown co-finding cannot be matched: the study leaves this family\n            code = np.zeros(len(pool), np.int64)\n            for arr in cof:\n                code = code * 8 + (np.asarray(arr, np.int64) + 1)\n            sig[:, c] = code\n            ok = v >= 0\n            pw[ok, c] = np.asarray(state_spec["prior"][l], np.float32)[v[ok]]\n            pair_fam.append(c)\n        pair_info = {"role": role, "sig": sig}\n        role_t, sig_t, pw_t = torch.from_numpy(role).to(dev), torch.from_numpy(sig).to(dev), torch.from_numpy(pw).to(dev)\n        print("pair families: " + "; ".join("%s pos %d neg %d" % (LABELS[c], int((role[:, c] == 1).sum()), int((role[:, c] == -1).sum()))\n                                              for c in pair_fam), flush=True)\n\n    ext_y = None\n    if a.extent:\n        if state_spec is None:\n            raise SystemExit("--extent needs --state-targets")\n        ext_y = torch.full((len(pool), C), float("nan"))\n        for l in ("Medial OA", "Lateral OA", "PF OA"):\n            c, cats_ = LABELS.index(l), state_spec["categories"][l]\n            pos_i, neg_i = cats_.index("high_grade_large"), {cats_.index("normal"), cats_.index("low_grade_chondral")}\n            si = st_idx[:, c].cpu()\n            ext_y[si == pos_i, c] = 1.0\n            for k_ in neg_i:\n                ext_y[si == k_, c] = 0.0\n        ext_y = ext_y.to(dev)\n        print("extent supervision: positives (>= 1 cm^2 stated) %s, negatives %s" % (\n            [int((ext_y[:, LABELS.index(l)] == 1).sum()) for l in ("Medial OA", "Lateral OA", "PF OA")],\n            [int((ext_y[:, LABELS.index(l)] == 0).sum()) for l in ("Medial OA", "Lateral OA", "PF OA")]), flush=True)\n\n    def extent_loss(area, sel):\n        """>= 1 cm^2 stated (1) vs normal / low grade (0), weighted by the report certainty like the state loss (audit 5)."""\n        y = ext_y[sel]\n        known = ~torch.isnan(y)\n        if not bool(known.any()):\n            return area.new_zeros(())\n        w_ = st_w[sel].float()[known]\n        logit_ = 2.0 * (torch.log1p(area.float()) - math.log(101.0))\n        l_ = F.binary_cross_entropy_with_logits(logit_[known], y[known], reduction="none")\n        return (l_ * w_).sum() / w_.sum().clamp_min(1e-6)\n\n    def pair_buckets(uids):\n        """(positive dataset indices, negative dataset indices, probability) per (family class, signature) with both\n        present; every family gets the same total probability, buckets within it ~ sqrt(n_pos * n_neg)."""\n        idx = {u: i for i, u in enumerate(pool)}\n        pi = np.array([idx[u] for u in uids])\n        out = []\n        for c in pair_fam:\n            r, sg = pair_info["role"][pi, c], pair_info["sig"][pi, c]\n            fam = []\n            for v in np.unique(sg[r != 0]):\n                P_ = np.flatnonzero((r == 1) & (sg == v)); N_ = np.flatnonzero((r == -1) & (sg == v))\n                if len(P_) and len(N_):\n                    fam.append((torch.from_numpy(P_), torch.from_numpy(N_), math.sqrt(len(P_) * len(N_))))\n            tot = sum(b[2] for b in fam)\n            out += [(p_, n_, w_ / tot / len(pair_fam)) for p_, n_, w_ in fam]\n        return out\n\n    def pair_loss(z, sel):\n        """Matched-pair ranking: for each ordered (positive i, hard negative j) of a family class in the batch with the same\n        co-finding signature, w * softplus(z_j - z_i), w = (prior_i - prior_j) * min(sw_i, sw_j); mean over active classes."""\n        r, sg, pr, cw = role_t[sel], sig_t[sel], pw_t[sel], st_w[sel]\n        zf = z.float()\n        tot, cnt = zf.new_zeros(()), 0\n        for c in pair_fam:\n            P_ = torch.nonzero(r[:, c] == 1).flatten()\n            N_ = torch.nonzero(r[:, c] == -1).flatten()\n            if len(P_) == 0 or len(N_) == 0:\n                continue\n            same = (sg[P_, c][:, None] == sg[N_, c][None, :]).float()\n            w = (pr[P_, c][:, None] - pr[N_, c][None, :]).clamp_min(0) * torch.minimum(cw[P_, c][:, None], cw[N_, c][None, :]) * same\n            if float(w.sum()) <= 0:\n                continue\n            # mean over the valid pairs with w as each pair\'s weight (dividing by sum(w) would cancel w for a single pair)\n            tot = tot + (w * F.softplus(zf[N_, c][None, :] - zf[P_, c][:, None])).sum() / same.sum()\n            cnt += 1\n        return tot / max(cnt, 1), cnt\n\n    if a.check_targets:\n        for item in a.arms.split(","):\n            kind, t, w = arm_tensors(item.split("=")[0])\n            print("check-targets arm %s (%s): %d x %d cells, finite %s, weighted cells %d, mean target per class %s" % (\n                item.split("=")[0], kind, t.shape[0], t.shape[1], bool(torch.isfinite(t).all()), int((w > 0).sum()),\n                [round(float(v), 3) for v in t.mean(0)]), flush=True)\n        print("check-targets: bias0 finite %s; holdout fold %s has %d pool studies" % (\n            bool(torch.isfinite(bias0).all()), a.holdout_fold, int((pool_fold == a.holdout_fold).sum()) if a.holdout_fold is not None else 0))\n        bad = []\n        for item in a.arms.split(","):\n            kind, t, w = arm_tensors(item.split("=")[0])\n            if not bool(torch.isfinite(t).all()):\n                bad.append("non-finite targets in arm " + item)\n            if kind == "lr" and bool((t <= 0).any()):\n                bad.append("non-positive likelihood ratios in arm " + item)\n            if kind == "bce" and bool(((t < 0) | (t > 1)).any()):\n                bad.append("probability targets outside [0, 1] in arm " + item)\n        if not bool(torch.isfinite(bias0).all()):\n            bad.append("non-finite bias0 (a blank or non-numeric prior in the pool)")\n        if state_spec is not None:\n            kc = torch.tensor([len(state_spec["categories"][l]) for l in LABELS], device=st_idx.device)\n            if bool((st_idx >= kc[None]).any()):\n                bad.append("state index outside its class\'s category list")\n        if bad:\n            print("check-targets FAILED: " + "; ".join(bad), flush=True)\n            raise SystemExit(1)\n        print("check-targets OK", flush=True)\n        raise SystemExit(0)\n\n    def rare_weights(uids):\n        """1 + 2[MCL state>=2 or acute] + 2[fracture state>=1] + [PF OA high] + [Baker>=moderate] + [traumatic edema/contusion], cap 4."""\n        idx = {u: i for i, u in enumerate(pool)}\n        L = {l: c for c, l in enumerate(LABELS)}\n        bn = {(b, l): j for j, (b, l) in enumerate(bin_names)}\n        w = []\n        for u in uids:\n            i = idx[u]\n            st = st_idx[i].tolist()\n            v = 1.0\n            v += 2.0 * (st[L["MCL"]] >= 2 or (("acute", "MCL") in bn and float(bin_t[i, bn[("acute", "MCL")]]) == 1.0))\n            v += 2.0 * (st[L["Fracture"]] >= 1)\n            v += 1.0 * (st[L["PF OA"]] in (3, 4))\n            v += 1.0 * (st[L["Baker\'s"]] in (3, 4, 5))\n            v += 1.0 * (st[L["Contusion"]] in (4, 5))\n            w.append(min(v, 4.0))\n        return torch.tensor(w, dtype=torch.double)\n\n    def make_loader(uids, mode, seed, shuffle):\n        ds = Studies(a.px, uids, a.train_windows, mode, seed, eval_stride=a.eval_stride,\n                     train_sampling=a.train_sampling, cover_stride=a.cover_stride, px_extra=a.px_extra, slices=a.slices,\n                     slice_jitter=a.slice_jitter if mode == "train" else 0.0, phys_mm=a.phys_mm, px_override=a.px_override,\n                     extent_meta=a.extent)\n        train = mode == "train"                     # evaluation: one study per batch, few workers, no pinned copies\n        sampler = None\n        if train and a.rare_boost:\n            if state_spec is None:\n                raise SystemExit("--rare-boost needs --state-targets")\n            w = rare_weights(uids)\n            sampler = torch.utils.data.WeightedRandomSampler(w, num_samples=len(uids), replacement=True,\n                                                             generator=torch.Generator().manual_seed(seed + 7))\n            shuffle = False\n            print("rare-boost: mean weight %.2f, boosted studies %d of %d" % (float(w.mean()), int((w > 1).sum()), len(w)), flush=True)\n        if train and a.pair_rank > 0:               # matched pairs on top of the same base distribution\n            bk = pair_buckets(uids)\n            bs = PairBatchSampler(len(uids), a.batch, None if sampler is None else sampler.weights, bk, a.pair_frac, seed + 7)\n            print("pair-rank: %d buckets over %d families, %.0f%% of batches drawn as a matched pair (others: base draws, weighted with replacement)" % (\n                len(bk), len(pair_fam), 100 * a.pair_frac), flush=True)\n            return ds, torch.utils.data.DataLoader(ds, batch_sampler=bs, num_workers=a.workers, collate_fn=collate,\n                                                   persistent_workers=False, pin_memory=True, prefetch_factor=2,\n                                                   generator=torch.Generator().manual_seed(seed))\n        return ds, torch.utils.data.DataLoader(ds, batch_size=a.batch if train else 1, shuffle=shuffle, sampler=sampler,\n                                               num_workers=a.workers if train else min(a.workers, a.eval_workers),\n                                               collate_fn=collate, drop_last=train, persistent_workers=False,\n                                               pin_memory=train, prefetch_factor=2 if train else 1,\n                                               generator=torch.Generator().manual_seed(seed))\n\n    def augment(img, gen):\n        B, W, _, S, _ = img.shape\n        oy = int(torch.randint(0, S - a.crop + 1, (1,), generator=gen))\n        ox = int(torch.randint(0, S - a.crop + 1, (1,), generator=gen))\n        img = img[..., oy:oy + a.crop, ox:ox + a.crop]\n        if a.aug != "v2":\n            return img\n        # per-study rotation (+-10 deg) and scale (0.9-1.1) through one affine grid; gamma 0.8-1.25; contrast 0.9-1.1.\n        # No flips: a horizontal flip swaps medial/lateral, a vertical one anterior/posterior.\n        Cc = img.shape[2]\n        x = img.float().reshape(B * W, Cc, a.crop, a.crop)\n        ang = (torch.rand(B, generator=gen) * 20 - 10) * math.pi / 180\n        sc = torch.rand(B, generator=gen) * 0.2 + 0.9\n        cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc\n        theta = torch.stack([torch.stack([cos, -sin, torch.zeros(B)], 1), torch.stack([sin, cos, torch.zeros(B)], 1)], 1)\n        theta = theta.repeat_interleave(W, 0).to(x.device)\n        grid = F.affine_grid(theta, x.shape, align_corners=False)\n        x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n        if a.aug_res > 0:                       # per-study in-plane resolution loss (coarse scanners: 0.33-0.5 mm pixels)\n            hit = torch.rand(B, generator=gen) < a.aug_res\n            fac = torch.rand(B, generator=gen) * 0.75 + 1.25\n            if bool(hit.any()):\n                xs = x.view(B, W, Cc, a.crop, a.crop)\n                for b in torch.nonzero(hit).flatten().tolist():\n                    s_ = max(8, int(round(a.crop / float(fac[b]))))\n                    lo = F.interpolate(xs[b], size=(s_, s_), mode="area")\n                    xs[b] = F.interpolate(lo, size=(a.crop, a.crop), mode="bilinear", align_corners=False)\n                x = xs.view(B * W, Cc, a.crop, a.crop)\n        gamma = (torch.rand(B, generator=gen) * 0.45 + 0.8).repeat_interleave(W, 0).to(x.device).view(-1, 1, 1, 1)\n        contrast = (torch.rand(B, generator=gen) * 0.2 + 0.9).repeat_interleave(W, 0).to(x.device).view(-1, 1, 1, 1)\n        x = (x / 255.).clamp_(0, 1).pow(gamma)\n        x = ((x - 0.5) * contrast + 0.5).clamp_(0, 1) * 255.\n        return x.round_().to(torch.uint8).reshape(B, W, Cc, a.crop, a.crop)\n\n    def center(img):\n        S = img.shape[-1]\n        o = (S - a.crop) // 2\n        return img[..., o:o + a.crop, o:o + a.crop]\n\n    @torch.no_grad()                                  # evaluation only: no autograd graph (same outputs, less time and memory)\n    def predict(model, uids, seed):\n        model.eval()\n        ds, dl = make_loader(uids, "eval", seed, False)\n        out = torch.zeros((len(uids), C), device=dev)\n        for img, meta, mask, idx in dl:\n            with torch.autocast("cuda", dtype=torch.bfloat16):\n                z = model(center(img.to(dev, non_blocking=True)), meta.to(dev), mask.to(dev), chunk=PREDICT_CHUNK)\n            out[idx.to(dev)] = z.float()\n        return out\n\n    @torch.no_grad()\n    def bn_recalibrate(model, uids, seed, n_batches):\n        """--ema copies only the trainable parameters into the shipped model, so BatchNorm running statistics still describe\n        the live weights (CoAtNet screen 10-08: holdout 0.61, predictions nearly constant).  Recompute them for the shipped\n        weights: cumulative averages over n_batches training batches, centre crop, no augmentation.  The whole model is in\n        train mode so that no zero padding enters the statistics (Net.forward pads only in eval mode or without BatchNorm);\n        the head\'s dropout comes after the encoder and does not reach them.  Returns the number of windows seen."""\n        bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]\n        if not bns or n_batches <= 0:\n            return 0\n        mom = [m.momentum for m in bns]\n        for m in bns:\n            m.reset_running_stats()\n            m.momentum = None\n        model.train()\n        ds_r, dl_r = make_loader(uids, "train", seed, True)\n        ds_r.slice_jitter = 0.0                      # audit 10-08 defect 1: no slice-jitter augmentation in the statistics\n        seen = used = 0\n        stance = torch.compiler.set_stance("force_eager") if hasattr(torch.compiler, "set_stance") else contextlib.nullcontext()\n        with stance:                                 # audit defect 2: never through a compiled encoder (guards, recompiles)\n            for i, (img, meta, mask, idx) in enumerate(dl_r):\n                if i >= n_batches:\n                    break\n                with torch.autocast("cuda", dtype=torch.bfloat16):\n                    model(center(img.to(dev, non_blocking=True)), meta.to(dev), mask.to(dev))\n                seen += int(mask.sum())\n                used += 1\n        for m, v in zip(bns, mom):\n            m.momentum = v\n        model.eval()\n        return used, seen\n\n    def reeval(arm, seed):\n        only = None if not a.only_folds else {int(x) for x in a.only_folds.split(",")}\n        pre = "partial_" if only else ""\n        path = os.path.join(a.out, pre + "preds_%s_s%d.json" % (arm, seed))\n        if os.path.exists(path):\n            return\n        src = json.load(open(os.path.join(a.reeval_run, "pool_uids.json"), encoding="utf-8"))\n        if src != pool:\n            raise SystemExit("pool of --reeval-run differs from the current pool")\n        z_gold = torch.zeros((len(gold), C), device=dev)\n        z_pool = torch.zeros((len(pool), C), device=dev)\n        t_arm = time.time()\n        H = a.holdout_fold\n        for k in range(K):\n            if only is not None and k not in only:\n                continue\n            if H is not None and k == H:                # no model is trained for the holdout fold\n                continue\n            ho = np.flatnonzero(pool_fold == k)\n            model = Net(a.backbone, a.init, freeze_blocks=a.freeze_blocks, crop=a.crop, state_spec=state_spec, in_chans=a.slices, spatial=a.spatial_attn, extent=a.extent).to(dev).to(memory_format=torch.channels_last)\n            sd = torch.load(os.path.join(a.reeval_run, "models", "%s_s%d_f%d.pt" % (arm, seed, k)), map_location="cpu")\n            if a.backbone == OFL_BACKBONE:\n                model.load_deploy_state_dict(sd)\n            else:\n                model.load_state_dict(sd)\n            if model.has_bn and a.bn_recal_batches > 0:\n                tr = np.flatnonzero((pool_fold != k) & (pool_fold != H)) if H is not None else np.flatnonzero(pool_fold != k)\n                b_, w_ = bn_recalibrate(model, [pool[i] for i in tr], 9100 + 17 * seed + k, a.bn_recal_batches)\n                print("  reeval fold %d: BatchNorm statistics recalibrated over %d batches (%d windows)" % (k, b_, w_), flush=True)\n            z_gold += predict(model, gold, seed) / (K if H is None else K - 1)\n            z_pool[torch.from_numpy(ho).to(dev)] = predict(model, [pool[i] for i in ho], seed)\n            if H is not None:                       # the clean holdout fold, as in training (mean of the fold models)\n                hh = np.flatnonzero(pool_fold == H)\n                z_pool[torch.from_numpy(hh).to(dev)] += predict(model, [pool[i] for i in hh], seed) / (K - 1)\n            print("  reeval %s seed %d fold %d done (eval stride %d) %.0fs" % (arm, seed, k, a.eval_stride, time.time() - t_arm), flush=True)\n            del model\n            torch.cuda.empty_cache()\n        np.save(os.path.join(a.out, pre + "oof_%s_s%d.npy" % (arm, seed)), z_pool.cpu().numpy().astype(np.float32))\n        out = {"arm": arm, "seed": seed, "reeval_of": a.reeval_run, "eval_stride": a.eval_stride, "gold_uids": gold, "labels": LABELS,\n               "holdout_fold": H,\n               "only_folds": sorted(only) if only else None,\n               "gold_fold": [int(by_uid[u]["fold"]) for u in gold],\n               "gold_logits": [[round(float(x), 5) for x in r] for r in z_gold.cpu().numpy()],\n               "receipt": [{"arm": arm, "kind": "reeval", "pool_studies": len(pool), "gold_in_training": 0, "trained": False}]}\n        json.dump(out, open(path, "w"))\n\n    def run(arm, seed):\n        if a.reeval_run:\n            return reeval(arm, seed)\n        only = None if not a.only_folds else {int(x) for x in a.only_folds.split(",")}\n        path = os.path.join(a.out, ("partial_" if only else "") + "preds_%s_s%d.json" % (arm, seed))\n        if os.path.exists(path):\n            return\n        kind, tgt, wgt = arm_tensors(arm)\n        z_gold = torch.zeros((len(gold), C), device=dev)\n        z_pool = torch.zeros((len(pool), C), device=dev)\n        logs, steps_total = [], 0\n        t_arm = time.time()\n        H = a.holdout_fold\n        for k in range(K):\n            if only is not None and k not in only:\n                print("  fold %d skipped (--only-folds)" % k, flush=True)\n                continue\n            if H is not None and k == H:\n                print("  fold %d is the clean holdout: no model trained for it" % k, flush=True)\n                continue\n            tr = np.flatnonzero((pool_fold != k) & (pool_fold != H)) if H is not None else np.flatnonzero(pool_fold != k)\n            ho = np.flatnonzero(pool_fold == k)\n            torch.manual_seed(1000 + 17 * seed)          # same init for every fold: heads start identical, so fold models can be souped\n            gen = torch.Generator().manual_seed(5000 + 17 * seed + k)\n            model = Net(a.backbone, a.init, freeze_blocks=a.freeze_blocks, crop=a.crop, state_spec=state_spec, in_chans=a.slices, spatial=a.spatial_attn, extent=a.extent).to(dev).to(memory_format=torch.channels_last)\n            torch.manual_seed(1000 + 17 * seed + k)      # fold-specific stream for dropout / drop path / augmentation\n            if a.grad_ckpt:                              # 32 GB cards: recompute encoder activations in backward (same maths, ~1.3x compute)\n                model.enc.set_grad_checkpointing(True)\n            if a.compile:                                # fused kernels; Module.compile keeps state_dict keys (checkpoints, deploy unchanged)\n                import torch._dynamo as _dynamo          # aliased: a bare \'import torch...\' here would make torch a local name\n                _dynamo.config.cache_size_limit = max(32, _dynamo.config.cache_size_limit)\n                model.enc.compile(dynamic=True)\n            with torch.no_grad():\n                model.bias.copy_(bias0)\n            enc = [p for n, p in model.named_parameters() if n.startswith("enc.") and p.requires_grad]\n            head = [p for n, p in model.named_parameters() if not n.startswith("enc.")]\n            groups = [{"params": enc, "lr": a.lr}]\n            vit_mod = model.enc.vit if a.backbone == OFL_BACKBONE else (model.enc if hasattr(model.enc, "blocks") else None)\n            if vit_mod is not None and a.llrd != 1.0:\n                nb = len(vit_mod.blocks)\n                groups, seen = [], set()\n                for i, blk in enumerate(vit_mod.blocks):\n                    ps = [p for p in blk.parameters() if p.requires_grad]\n                    if ps:\n                        groups.append({"params": ps, "lr": a.lr * (a.llrd ** (nb - 1 - i))}); seen.update(id(p) for p in ps)\n                rest = [p for p in enc if id(p) not in seen]          # final norm (top) and, when unfrozen, embeddings (bottom)\n                top = [p for n, p in vit_mod.named_parameters() if n.startswith("norm.") and p.requires_grad]\n                bottom = [p for p in rest if id(p) not in {id(q) for q in top}]\n                if top:\n                    groups.append({"params": top, "lr": a.lr})\n                if bottom:\n                    groups.append({"params": bottom, "lr": a.lr * (a.llrd ** nb)})\n            if k == (sorted(only)[0] if only else 0) and seed == 0:\n                print("trainable encoder params %.1fM, head params %.2fM, frozen encoder params %.1fM" % (\n                    sum(p.numel() for p in enc) / 1e6, sum(p.numel() for p in head) / 1e6,\n                    sum(p.numel() for n, p in model.named_parameters() if n.startswith("enc.") and not p.requires_grad) / 1e6), flush=True)\n            opt = torch.optim.AdamW(groups + [{"params": head, "lr": a.head_lr}], weight_decay=a.wd)\n            ema = None\n            if a.ema > 0:                               # EMA over everything that trains (frozen parts are shared anyway)\n                ema = {n_: p_.detach().clone() for n_, p_ in model.named_parameters() if p_.requires_grad}\n            ds, dl = make_loader([pool[i] for i in tr], "train", 9000 + 17 * seed + k, True)\n            total = len(dl) * a.epochs\n            warm = max(1, int(.05 * total))\n            sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else\n                                                      .5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))\n            tr_t = torch.from_numpy(tr).to(dev)\n            log = []\n            # resumable state, written atomically at the end of every --ckpt-every epochs: weights, optimizer, LR schedule, EMA,\n            # and every random stream that decides the next epoch (augmentation, sampler, loader, CPU and GPU global streams)\n            ckpt_path = os.path.join(a.out, "ckpt", "%s_s%d_f%d.pt" % (arm, seed, k))\n            start_ep = 0\n            if a.resume and os.path.exists(ckpt_path):\n                ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)\n                if ck.get("epochs") != a.epochs or ck.get("train_studies") != len(tr):\n                    raise SystemExit("checkpoint %s does not match this run (epochs / training set)" % ckpt_path)\n                model.load_state_dict(ck["model"])\n                opt.load_state_dict(ck["opt"])\n                sched.load_state_dict(ck["sched"])\n                if ema is not None:\n                    ema = {n_: v.to(dev) for n_, v in ck["ema"].items()}\n                gen.set_state(ck["gen"])\n                if ck.get("sampler_gen") is not None:\n                    (dl.batch_sampler.generator if isinstance(dl.batch_sampler, PairBatchSampler) else dl.sampler.generator).set_state(ck["sampler_gen"])\n                if ck.get("loader_gen") is not None:\n                    dl.generator.set_state(ck["loader_gen"])\n                torch.set_rng_state(ck["cpu_rng"])\n                torch.cuda.set_rng_state_all(ck["cuda_rng"])\n                log = list(ck["log"])\n                start_ep = int(ck["epoch"]) + 1\n                print("  resumed %s seed %d fold %d after epoch %d (%s)" % (arm, seed, k, ck["epoch"], ckpt_path), flush=True)\n            for ep in range(start_ep, a.epochs):\n                ds.epoch = ep\n                model.train()\n                tot = n = 0\n                pr_batches = pr_terms = 0\n                ext_acc = torch.zeros(6, device=dev)\n                for img, meta, mask, idx in dl:\n                    sel = tr_t[idx.to(dev)]\n                    with torch.autocast("cuda", dtype=torch.bfloat16):\n                        z, sl, bl = model(augment(img.to(dev, non_blocking=True), gen), meta.to(dev), mask.to(dev), return_aux=True)\n                    loss = bce_soft(z, tgt[sel], wgt[sel]) if kind == "bce" else lr_loss(z, tgt[sel])\n                    if state_spec is not None:\n                        loss = loss + a.state_weight * state_loss(sl, bl, st_idx[sel], st_w[sel], bin_t[sel])\n                    if a.extent:\n                        ext_l = extent_loss(model._ext_area, sel)\n                        if a.extent_weight > 0:\n                            loss = loss + a.extent_weight * ext_l\n                        with torch.no_grad():                     # epoch log of the extent path (audit 6), no sync per step\n                            la_, y_ = torch.log1p(model._ext_area.float() / 100.0), ext_y[sel]\n                            ext_acc += torch.stack([(la_ * (y_ == 1)).sum(), (y_ == 1).sum().float(), (la_ * (y_ == 0)).sum(),\n                                                    (y_ == 0).sum().float(), ext_l.detach().float(), torch.ones((), device=dev)])\n                    if a.pair_rank > 0:\n                        pl, pc = pair_loss(z, sel)\n                        loss = loss + a.pair_rank * pl\n                        pr_batches += int(pc > 0); pr_terms += pc\n                    opt.zero_grad(set_to_none=True)\n                    loss.backward()\n                    if a.forward_only:\n                        g = sum(float(p.grad.abs().sum()) for p in model.parameters() if p.grad is not None)\n                        print("forward-only check: loss %.4f, gradient mass %.3e, windows %d, optimizer steps 0" % (\n                            float(loss), g, int(mask.sum())), flush=True)\n                        return\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)\n                    opt.step()\n                    sched.step()\n                    if ema is not None:\n                        with torch.no_grad():\n                            d_ = min(a.ema, (1 + sched.last_epoch) / (10 + sched.last_epoch))   # warm EMA early on\n                            for n_, p_ in model.named_parameters():\n                                if n_ in ema:\n                                    ema[n_].mul_(d_).add_(p_.detach(), alpha=1 - d_)\n                    tot += float(loss.detach()) * len(idx)\n                    n += len(idx)\n                log.append(round(tot / max(1, n), 5))\n                print("  %s seed %d fold %d epoch %d loss %.4f  %.0fs" % (arm, seed, k, ep, log[-1], time.time() - t_arm), flush=True)\n                if a.extent:\n                    ea = ext_acc.tolist()\n                    print("  extent epoch %d: auxiliary loss %.4f (weight %.2f) | gamma MOA/LOA/PFOA %s | mean log1p(A/100) on >= 1 cm^2 stated %.3f vs normal/low grade %.3f" % (\n                        ep, ea[4] / max(1.0, ea[5]), a.extent_weight,\n                        [round(float(model.ext_gamma[LABELS.index(l)]), 3) for l in ("Medial OA", "Lateral OA", "PF OA")],\n                        ea[0] / max(1.0, ea[1]), ea[2] / max(1.0, ea[3])), flush=True)\n                if a.pair_rank > 0:\n                    print("  pair-rank epoch %d: %d of %d batches had at least one matched-pair term (%d class terms; includes base batches that happened to pair)" % (ep, pr_batches, len(dl), pr_terms), flush=True)\n                if a.ckpt_every > 0 and (ep + 1) % a.ckpt_every == 0 and ep + 1 < a.epochs:\n                    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)\n                    sgen = getattr(dl.sampler, "generator", None) if isinstance(dl.sampler, torch.utils.data.WeightedRandomSampler) else None\n                    if isinstance(dl.batch_sampler, PairBatchSampler):\n                        sgen = dl.batch_sampler.generator\n                    torch.save({"epoch": ep, "epochs": a.epochs, "train_studies": len(tr), "log": log,\n                                "model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),\n                                "ema": None if ema is None else {n_: v.detach().cpu() for n_, v in ema.items()},\n                                "gen": gen.get_state(), "sampler_gen": None if sgen is None else sgen.get_state(),\n                                "loader_gen": None if dl.generator is None else dl.generator.get_state(),\n                                "cpu_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all()}, ckpt_path + ".tmp")\n                    os.replace(ckpt_path + ".tmp", ckpt_path)\n                if a.stop_after_epoch >= 0 and ep == a.stop_after_epoch:\n                    print("  stop-after-epoch %d: exiting (test)" % ep, flush=True)\n                    sys.exit(3)\n            steps_total += total\n            if ema is not None:                         # evaluate and ship the EMA weights\n                with torch.no_grad():\n                    for n_, p_ in model.named_parameters():\n                        if n_ in ema:\n                            p_.copy_(ema[n_])\n                if model.has_bn and a.bn_recal_batches > 0:\n                    b_, w_ = bn_recalibrate(model, [pool[i] for i in tr], 9100 + 17 * seed + k, a.bn_recal_batches)\n                    print("  BatchNorm statistics recalibrated for the EMA weights: %d batches, %d windows" % (b_, w_), flush=True)\n                elif model.has_bn:\n                    print("  WARNING: BatchNorm backbone with --ema and no --bn-recal-batches: the saved model pairs EMA weights with "\n                          "the live weights\' BatchNorm statistics (see the 10-08 CoAtNet screen)", flush=True)\n            os.makedirs(os.path.join(a.out, "models"), exist_ok=True)\n            torch.save(model.deploy_state_dict(),\n                       os.path.join(a.out, "models", "%s_s%d_f%d.pt" % (arm, seed, k)))\n            if os.path.exists(ckpt_path):                 # the fold is complete: its resumable state is no longer needed\n                os.remove(ckpt_path)\n            z_gold += predict(model, gold, seed) / (K if H is None else K - 1)\n            z_pool[torch.from_numpy(ho).to(dev)] = predict(model, [pool[i] for i in ho], seed)\n            if H is not None:\n                hh = np.flatnonzero(pool_fold == H)\n                z_pool[torch.from_numpy(hh).to(dev)] += predict(model, [pool[i] for i in hh], seed) / (K - 1)\n            logs.append([log[0], log[-1]])\n            del model, opt\n            torch.cuda.empty_cache()\n        np.save(os.path.join(a.out, ("partial_" if only else "") + "oof_%s_s%d.npy" % (arm, seed)), z_pool.cpu().numpy().astype(np.float32))\n        out = {"arm": arm, "seed": seed, "epochs": a.epochs, "nested": False, "gold_uids": gold, "only_folds": sorted(only) if only else None,\n               "holdout_fold": H,\n               "gold_fold": [int(by_uid[u]["fold"]) for u in gold], "labels": LABELS,\n               "gold_logits": [[round(float(x), 5) for x in r] for r in z_gold.cpu().numpy()],\n               "loss_first_last": logs, "optimizer_steps": steps_total, "train_seconds": round(time.time() - t_arm, 1),\n               "receipt": [{"arm": arm, "kind": kind, "pool_studies": len(pool), "gold_in_training": 0,\n                            "init_seed": 1000 + 17 * seed, "note": "fold models share one initialisation (soup-compatible)"}]}\n        json.dump(out, open(path, "w"))\n\n    cfg = {k: getattr(a, k) for k in vars(a)}\n    cfg.update({"pool_studies": len(pool), "gold_studies": len(gold), "feature_studies": len(done), "labels": LABELS,\n                "scaffold": "compact fine-tuned image model on the pixel cache"})\n    json.dump(cfg, open(os.path.join(a.out, "run_config.json"), "w"), indent=1)\n    json.dump(pool, open(os.path.join(a.out, "pool_uids.json"), "w"))\n    for seed in range(a.seed_base, a.seed_base + a.seeds):\n        for arm in [x.split("=", 1)[0] for x in a.arms.split(",") if x]:\n            run(arm, seed)\n            if a.forward_only:\n                return\n    print("ALL DONE", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'compact_infer.py': '"""Compact-model inference worker (Kaggle or local): raw DICOM -> image scores for one list of studies.\n\nPath, identical to how the compact model was trained and evaluated:\n  raw DICOM  --mainline training.native_cache.prepare (one study at a time)-->  native uint16 series\n             --px_cache.StudySlices.build (the SAME file that built the pixel cache)-->  288 px slices\n             --compact_train evaluation windows (every third slice as centre, both views, 3 adjacent slices)-->\n             --centre crop 256, fold models of compact_train.Net, mean logit-->  12 probabilities\nNothing is trained here.  No report, no label, no test-time fitting.  Failed or unreached studies are\nlisted as fallbacks for the controller.\n"""\nimport argparse, contextlib, csv, hashlib, io, json, os, shutil, sys, time\nfrom pathlib import Path\nsys.dont_write_bytecode = True\nHERE = Path(__file__).resolve().parent\nimport numpy as np\nimport torch\n\nUIDK = "StudyInstanceUID"\nFORCED_SIDE = {}        # study uid -> "L" / "R": the native render uses this side instead of tags / sagittal IPP x\n\n\nclass CStudies(torch.utils.data.Dataset):\n    def __init__(self, data, split, uids, tmp, code, size, eval_stride=3, views="v1", slices=3, anchor_weights=None, phys_mm=0.0,\n                 side_model=None, side_threshold=2.0, side_max_x=0.0, side_position_mm=20.0, extent_meta=False):\n        self.data, self.split, self.uids = str(data), split, list(uids)\n        # extent (M4) fold models read three more meta columns: series index, plane, slices in the series -- the same cache\n        # columns (0, 2, 10) compact_train.Studies appends with extent_meta\n        self.extent_meta = bool(extent_meta)\n        self.tmp, self.code, self.size = str(tmp), str(code), int(size)\n        self.eval_stride, self.views, self.slices = int(eval_stride), str(views), int(slices)\n        self.phys_mm = float(phys_mm)\n        self.anchor_weights = None if anchor_weights is None else str(anchor_weights)\n        if self.views in ("v3k", "v4") and not self.anchor_weights:\n            raise ValueError("views v3k need --anchor-weights (SKM-TEA checkpoint) to rebuild the knee anchors at inference")\n        self.native = None\n        # image side check (bundles that ship a side model): studies without one explicit DICOM laterality tag are\n        # scored by the orientation classifier on their canonical coronal/axial slices; a confidently mirrored study is\n        # re-rendered with the other side (native render, geometry, anchors and views all follow).  None = off.\n        self.side_model = None if side_model is None else str(side_model)\n        self.side_threshold, self.side_max_x = float(side_threshold), float(side_max_x)\n        self.side_position_mm = float(side_position_mm)\n        self.side_net = None\n        self.source_directory = True     # False: read DICOMs from the chunked competition archive under <data>/download\n\n    def __len__(self):\n        return len(self.uids)\n\n    def _setup(self):\n        for p in (self.code, str(HERE)):\n            if p not in sys.path:\n                sys.path.insert(0, p)\n        import probe_infer as PI\n        self.PI = PI\n        self.native = PI.install_native(True)\n        import px_cache\n        import compact_train\n        self.builder = px_cache.StudySlices(self.tmp, [], self.size, self.views)   # (self.slices is the window depth K)\n        self.window_rows = compact_train.window_rows          # the trainer\'s own window list (eval mode)\n        self.bank = None\n        if self.views in ("v3k", "v4"):\n            # knee-centred views: per-series geometry from the DICOM headers (mainline geometry bank, the code that\n            # built the training geometry) + SKM-TEA joint anchors (knee_anchor_bank_v2, the code of the training bank)\n            import training.geometry_bank as GB\n            import knee_anchor_bank_v2 as KB\n            self.GB, self.KB = GB, KB\n            dev = "cuda" if torch.cuda.is_available() else "cpu"\n            if dev == "cpu":\n                torch.set_num_threads(2)\n            self.bank = KB.AnchorBank(KB.load_model(self.anchor_weights, dev), dev)\n        import training.dicom_geometry as G\n        if not getattr(G, "_forced_side_wrapped", False):\n            import dataclasses\n            orig = G.decide_study_side\n\n            def decide(study, tag_values, sagittal_x, _orig=orig):\n                r = _orig(study, tag_values, sagittal_x)\n                f = FORCED_SIDE.get(str(study))\n                return r if f is None else dataclasses.replace(r, side=f, source="image_side_check")\n            G.decide_study_side = decide\n            G._forced_side_wrapped = True\n        if self.side_model:\n            import side_common as SC\n            self.SC = SC\n            sdev = "cuda" if torch.cuda.is_available() else "cpu"\n            net = SC.build_side_net()\n            net.load_state_dict(torch.load(self.side_model, map_location="cpu", weights_only=True), strict=True)\n            self.side_net, self.side_dev = net.eval().to(sdev), sdev\n\n    def _prepare(self, uid, work):\n        with contextlib.redirect_stdout(io.StringIO()):\n            self.native.prepare(self.data, work, split=self.split, study_ids=[uid], source_directory=self.source_directory)\n        if self.bank is not None or self.side_net is not None:\n            with contextlib.redirect_stdout(io.StringIO()):\n                self.GB.build(root=self.data, manifest=str(work / "manifest.json"), output=str(work / "geometry"),\n                              split=self.split, source_directory=self.source_directory)\n\n    def _native_and_geometry(self, uid, work):\n        """native render + geometry bank for one study; with a side model, the image side check (may re-render)."""\n        self.PI.OneStudySource.current = uid\n        self._prepare(uid, work)\n        if self.side_net is None:\n            return None\n        key = hashlib.sha256(uid.encode()).hexdigest()[:24]\n        with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\n            study = json.load(fh)\n        sides = {s.get("side") for s in study["series"]}\n        cur = next(iter(sides)) if len(sides) == 1 else None\n        status = set()\n        for s in study["series"]:\n            gp = work / "geometry" / (hashlib.sha256((uid + "/" + s["SeriesInstanceUID"]).encode()).hexdigest()[:24] + ".json")\n            if gp.exists():\n                status.add(json.loads(gp.read_text(encoding="utf-8")).get("laterality_status"))\n        sag = [float(np.median(s["positions"])) for s in study["series"] if s.get("plane") == "Sagittal" and s.get("positions")]\n        x = float(np.median(sag)) if sag else None\n        note = {"side": cur, "status": "|".join(sorted(str(v) for v in status)), "x": None if x is None else round(x, 1),\n                "score": None, "flipped": False}\n        if status == {"explicit_study_tags"} and cur in ("L", "R"):\n            # one explicit DICOM laterality tag stands unless the knee lies clearly on the other side of the scanner\n            # (10-07: 18 training studies had the tag contradicted by both the position and the images)\n            pos = None if x is None or abs(x) < self.side_position_mm else ("L" if x >= 0 else "R")\n            if pos is None or pos == cur:\n                return note\n            note["tag_contradicted_by_position"] = True\n        if self.side_max_x > 0 and cur in ("L", "R") and x is not None and abs(x) >= self.side_max_x:\n            return note                                   # far from the scanner centre: the sagittal IPP x rule holds\n        ims = []\n        for s in study["series"]:\n            if s.get("plane") not in self.SC.SIDE_PLANES:\n                continue\n            p = Path(s["path"])\n            if not p.exists():\n                p = work / "series" / os.path.basename(str(s["path"]).replace(chr(92), "/"))\n            if p.exists():\n                ims.append(self.SC.side_inputs(np.load(p, mmap_mode="r")))\n        ims = [v for v in ims if len(v)]\n        if not ims:\n            return note\n        xb = torch.from_numpy(np.concatenate(ims)).float().div_(255.)[:, None]\n        with torch.inference_mode():\n            z = self.side_net(((xb - 0.45) / 0.25).to(self.side_dev)).float().squeeze(1)\n        score = float(z.mean())\n        note["score"] = round(score, 3)\n        if score < -self.side_threshold:\n            new = "R" if cur in (None, "L") else "L"\n            FORCED_SIDE[uid] = new\n            shutil.rmtree(work, ignore_errors=True)\n            self._prepare(uid, work)\n            with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\n                got = {s.get("side") for s in json.load(fh)["series"]}\n            if got != {new}:\n                raise ValueError("forced side %s not applied (%s)" % (new, got))\n            note.update(flipped=True, new_side=new)\n        return note\n\n    def rows(self, uid, forced_side=None):\n        """training-cache override builder: the same native -> geometry -> anchors -> views path for one study, with an\n        optional forced side; returns (StudySlices.build output, set of sides in the rendered study)."""\n        if self.native is None:\n            self._setup()\n        key = hashlib.sha256(uid.encode()).hexdigest()[:24]\n        work = Path(self.tmp) / ("r_" + key[:16])\n        if forced_side is not None:\n            FORCED_SIDE[uid] = forced_side\n        try:\n            self.PI.OneStudySource.current = uid\n            with contextlib.redirect_stdout(io.StringIO()):\n                self.native.prepare(self.data, work, split=self.split, study_ids=[uid], source_directory=self.source_directory)\n                self.GB.build(root=self.data, manifest=str(work / "manifest.json"), output=str(work / "geometry"),\n                              split=self.split, source_directory=self.source_directory)\n            self.builder.cache = str(work)\n            with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\n                study = json.load(fh)\n            if self.bank is not None:\n                rec = self.bank.record(uid, study, geo_dir=str(work / "geometry"))\n                (work / "anchors").mkdir(exist_ok=True)\n                with open(work / "anchors" / (self.KB.study_key(uid) + ".json"), "w", encoding="utf-8") as fh:\n                    json.dump(rec, fh)\n                self.builder.anchors = str(work / "anchors")\n            return self.builder.build(uid), {s.get("side") for s in study["series"]}\n        finally:\n            FORCED_SIDE.pop(uid, None)\n            shutil.rmtree(work, ignore_errors=True)\n\n    def __getitem__(self, i):\n        uid = self.uids[i]\n        t0 = time.perf_counter()\n        work = Path(self.tmp) / ("s_" + hashlib.sha256(uid.encode()).hexdigest()[:16])\n        t1 = t0\n        try:\n            if self.native is None:\n                self._setup()\n            side_note = self._native_and_geometry(uid, work)\n            t1 = time.perf_counter()\n            self.builder.cache = str(work)\n            if self.bank is not None:\n                key = self.KB.study_key(uid)\n                with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\n                    study = json.load(fh)\n                rec = self.bank.record(uid, study, geo_dir=str(work / "geometry"))\n                (work / "anchors").mkdir(exist_ok=True)\n                with open(work / "anchors" / (key + ".json"), "w", encoding="utf-8") as fh:\n                    json.dump(rec, fh)\n                self.builder.anchors = str(work / "anchors")\n                anchor_note = {"method": rec["method"], "why": rec["why"][:4]}\n            else:\n                anchor_note = None\n            t2 = time.perf_counter()\n            item = self.builder.build(uid)\n            px, meta = item["px"].numpy(), item["meta"].numpy()\n            blocks = [{"n_slices": b["n_slices"], "views": [(0, v, st) for v, st in b["views"]]} for b in item["blocks"]]\n            cand = self.window_rows(blocks, lambda f, r: meta[r], "eval", self.slices, self.eval_stride, phys_mm=self.phys_mm)\n            if not cand:\n                raise ValueError("no usable window")\n            rows = np.array(cand, np.int64)                 # (file, centre row, K slice rows)\n            img = np.stack([px[rows[:, 2 + d]] for d in range(self.slices)], 1)\n            m = meta[rows[:, 1]]\n            meta6 = np.stack([m[:, 1], m[:, 3], m[:, 4], m[:, 5], m[:, 6], m[:, 7]], 1).astype(np.float32)\n            if self.extent_meta:\n                meta6 = np.concatenate([meta6, np.stack([m[:, 0], m[:, 2], m[:, 10]], 1).astype(np.float32)], 1)\n            out = {"uid": uid, "error": None, "images": torch.from_numpy(img), "meta": torch.from_numpy(meta6), "anchor": anchor_note,\n                   "side": side_note}\n        except Exception as exc:\n            t2 = locals().get("t2", t1)\n            out = {"uid": uid, "error": repr(exc)[:300], "images": torch.zeros((0, self.slices, self.size, self.size), dtype=torch.uint8),\n                   "meta": torch.zeros((0, 9 if self.extent_meta else 6), dtype=torch.float32), "anchor": None}\n        finally:\n            shutil.rmtree(work, ignore_errors=True)\n        out["seconds"] = [round(t1 - t0, 3), round(time.perf_counter() - t2, 3), round(t2 - t1, 3)]   # decode, render, anchor\n        return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--split", default="test")\n    ap.add_argument("--code", required=True)\n    ap.add_argument("--models", required=True, help="folder with <arm>_s<seed>_f<fold>.pt state dicts")\n    ap.add_argument("--arm", required=True)\n    ap.add_argument("--backbone", default="resnet34")\n    ap.add_argument("--init", default=None, help="ofl_l backbone: the public OFL-L weights (safetensors) the fold models are rebuilt on")\n    ap.add_argument("--freeze-blocks", type=int, default=16)\n    ap.add_argument("--size", type=int, default=288)\n    ap.add_argument("--crop", type=int, default=256)\n    ap.add_argument("--views", default="v1", help="px_cache view set the fold models were trained with (v1 / v2)")\n    ap.add_argument("--state-spec", default=None, help="state_spec json of the report-state construct the fold models were trained with")\n    ap.add_argument("--slices", type=int, default=3, help="adjacent slices per window the fold models were trained with")\n    ap.add_argument("--anchor-weights", default=None, help="views v3k: the SKM-TEA checkpoint (anatomy.pt of the code dataset) for the knee anchors")\n    ap.add_argument("--phys-mm", type=float, default=0.0, help="window spacing in mm the fold models were trained with (compact_train --phys-mm)")\n    ap.add_argument("--eval-stride", type=int, default=3, help="window centres: every k-th slice (3 = training-time validation default; 1 = every slice)")\n    ap.add_argument("--assignment", default=None)\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--tmp", default=None)\n    ap.add_argument("--amp", default="auto", choices=["auto", "fp16", "bf16", "fp32"])\n    ap.add_argument("--chunk", type=int, default=192)\n    ap.add_argument("--workers", type=int, default=2)\n    ap.add_argument("--prefetch", type=int, default=1)\n    ap.add_argument("--item-timeout", type=float, default=900.)\n    ap.add_argument("--deadline", type=float, default=0.)\n    ap.add_argument("--dump-logits", default=None)\n    ap.add_argument("--models2", default=None, help="second model group (another architecture), run on the same windows")\n    ap.add_argument("--arm2", default=None, help="file prefix of the second group (default: --arm)")\n    ap.add_argument("--backbone2", default=None)\n    ap.add_argument("--w2", default=None, help="weight of group 2 per class: one number or 12 comma-separated (label order); "\n                    "p = (1 - w) p_group1 + w p_group2")\n    ap.add_argument("--side-model", default=None, help="orientation classifier (side_common.build_side_net state dict); off when absent")\n    ap.add_argument("--side-threshold", type=float, default=2.0, help="flip a study only when its mean logit < -threshold")\n    ap.add_argument("--side-max-x", type=float, default=0.0, help="> 0: skip the check for untagged studies this far (mm) from x = 0")\n    ap.add_argument("--side-position-mm", type=float, default=20.0, help="a DICOM tag is re-checked only when the knee lies this far on the other side")\n    a = ap.parse_args()\n    out = Path(a.output)\n    out.mkdir(parents=True, exist_ok=True)\n    tmp = Path(a.tmp or (out / "tmp"))\n    tmp.mkdir(parents=True, exist_ok=True)\n    t_start = time.time()\n    if a.assignment:\n        uids = json.loads(Path(a.assignment).read_text(encoding="utf-8"))\n    else:\n        with open(Path(a.data) / (a.split + ".csv"), encoding="utf-8") as fh:\n            uids = [r[UIDK] for r in csv.DictReader(fh)]\n    if len(uids) != len(set(uids)):\n        raise SystemExit("duplicate study ids")\n\n    dev = torch.device("cuda")\n    for p in (a.code, str(HERE)):\n        if p not in sys.path:\n            sys.path.insert(0, p)\n    import timm\n    import compact_train as CT\n    LABELS = CT.LABELS\n    cap = torch.cuda.get_device_capability()[0]\n    amp = a.amp if a.amp != "auto" else ("bf16" if cap >= 8 else "fp16")\n    state_spec = json.load(open(a.state_spec, encoding="utf-8")) if a.state_spec else None\n\n    def load_group(folder, arm, backbone):\n        fs = sorted(Path(folder).glob(arm + "_s*_f*.pt"))\n        if not fs:\n            raise SystemExit("no model files for arm %s in %s" % (arm, folder))\n        ms, n_sp = [], 0\n        for p in fs:\n            sd = torch.load(p, map_location="cpu", weights_only=True)\n            extent = any(k.split(".")[-1] in ("ext_u", "ext_view", "ext_gamma") for k in sd)   # M4 fold model (OA extent)\n            if extent:\n                N_EXTENT[0] += 1\n            spatial = any(k == "sp_q" or k.endswith(".sp_q") for k in sd)       # P2 fold model (per-class spatial attention)\n            n_sp += int(spatial)\n            kw = {}\n            if spatial:\n                kw["spatial"] = True\n            if extent:\n                kw["extent"] = True\n            m = CT.Net(backbone, a.init if backbone == CT.OFL_BACKBONE else None, freeze_blocks=a.freeze_blocks, crop=a.crop,\n                       state_spec=state_spec, in_chans=a.slices, **kw)\n            if backbone == CT.OFL_BACKBONE:\n                m.load_deploy_state_dict(sd)          # fold model = public weights + trained tail/head\n            else:\n                m.load_state_dict(sd, strict=True)\n            ms.append(m.eval().requires_grad_(False).to(dev).to(memory_format=torch.channels_last))\n        return fs, ms, n_sp\n\n    N_EXTENT = [0]\n    files, models, n_spatial = load_group(a.models, a.arm, a.backbone)\n    files2, models2, n_spatial2, W2 = [], [], 0, None\n    if a.models2:\n        if not a.backbone2 or a.backbone2 == CT.OFL_BACKBONE or a.backbone == CT.OFL_BACKBONE:\n            raise SystemExit("--models2 needs --backbone2 (timm backbones only, no OFL shared trunk)")\n        files2, models2, n_spatial2 = load_group(a.models2, a.arm2 or a.arm, a.backbone2)\n        w = [float(x) for x in (a.w2 or "0.5").split(",")]\n        w = w * len(LABELS) if len(w) == 1 else w\n        if len(w) != len(LABELS) or not all(0.0 <= x <= 1.0 for x in w):\n            raise SystemExit("--w2: one weight or %d weights in [0, 1]" % len(LABELS))\n        W2 = torch.tensor(w, dtype=torch.float64, device=dev)\n    shared_trunk = a.backbone == CT.OFL_BACKBONE and len(models) > 1\n    if shared_trunk:\n        # all fold models share the frozen trunk bit-for-bit; keep one copy and run it once per window batch\n        sd0 = models[0].enc.state_dict()\n        for m in models[1:]:\n            sdm = m.enc.state_dict()\n            for k in models[0].enc.frozen_keys:\n                if not torch.equal(sdm[k], sd0[k]):\n                    raise SystemExit("fold models do not share the frozen trunk: " + k)\n    print("MODEL_READY", json.dumps({"gpu": torch.cuda.get_device_name(), "amp": amp, "models": len(models),\n                                     "models2": len(models2), "spatial_models": n_spatial + n_spatial2, "extent_models": N_EXTENT[0],\n                                     "studies": len(uids), "seconds": round(time.time() - t_start, 1)}), flush=True)\n    off = (a.size - a.crop) // 2\n\n    def run_model(m, img, meta, mask, mode):\n        if mode == "fp32":\n            return m(img, meta, mask, chunk=a.chunk).float()\n        with torch.autocast("cuda", dtype=torch.float16 if mode == "fp16" else torch.bfloat16):\n            return m(img, meta, mask, chunk=a.chunk).float()\n\n    def heads_from_tokens(m, tok, meta, mask):\n        """compact_train.Net.head: encoder output -> class logits (the same code the trainer used, state construct included)."""\n        return m.head(tok, meta, mask)\n\n    def run_shared(img, meta, mask, mode):\n        """OFL fold models: frozen trunk once per window chunk, then every fold\'s trained tail + head."""\n        x = img[mask]\n        m0 = models[0]\n        tails = [[] for _ in models]\n        step = a.chunk or len(x)\n        ctx = contextlib.nullcontext() if mode == "fp32" else torch.autocast("cuda", dtype=torch.float16 if mode == "fp16" else torch.bfloat16)\n        with ctx:\n            for s0 in range(0, len(x), step):\n                xs = x[s0:s0 + step].float().div_(255.)\n                xs = ((xs - m0.mean) / m0.std).contiguous(memory_format=torch.channels_last)\n                tr = m0.enc.trunk(xs)\n                for i, m in enumerate(models):\n                    tails[i].append(m.enc.tail(tr))\n            zs = [heads_from_tokens(m, torch.cat(tails[i]), meta, mask) for i, m in enumerate(models)]\n        return torch.stack([z[0].float() for z in zs])\n\n    @torch.inference_mode()\n    def score(images, meta):\n        img = images.to(dev, non_blocking=True)[None, :, :, off:off + a.crop, off:off + a.crop]\n        meta = meta.to(dev)[None]\n        mask = torch.ones((1, img.shape[1]), dtype=torch.bool, device=dev)\n        if shared_trunk:\n            zs = run_shared(img, meta, mask, amp)\n            if not torch.isfinite(zs).all():\n                zs = run_shared(img, meta, mask, "fp32")\n            return zs.mean(0), zs\n        zs = []\n        for m in models:\n            z = run_model(m, img, meta, mask, amp)\n            if not torch.isfinite(z).all():\n                z = run_model(m, img, meta, mask, "fp32")\n            zs.append(z[0])\n        zs = torch.stack(zs)\n        if not models2:\n            return zs.mean(0), zs\n        zs2 = []\n        for m in models2:                                     # second architecture on the same windows (one decode)\n            z = run_model(m, img, meta, mask, amp)\n            if not torch.isfinite(z).all():\n                z = run_model(m, img, meta, mask, "fp32")\n            zs2.append(z[0])\n        zs2 = torch.stack(zs2)\n        p1, p2 = torch.sigmoid(zs.mean(0).double()), torch.sigmoid(zs2.mean(0).double())\n        pf = ((1 - W2) * p1 + W2 * p2).clamp(1e-12, 1 - 1e-12)\n        return torch.log(pf / (1 - pf)), torch.cat([zs, zs2])\n\n    fh = open(out / "predictions.csv", "w", encoding="utf-8", newline="")\n    wr = csv.writer(fh)\n    wr.writerow([UIDK] + LABELS)\n    fallback, stats = {}, {"decode_s": 0., "render_s": 0., "gpu_s": 0., "windows": 0, "wait_s": 0., "rows": 0, "anchor_s": 0.}\n    per_study, dump = [], {}\n    anchor_methods = {}\n    side_counts = {"checked": 0, "flipped": 0}\n\n    def handle(item):\n        uid = item["uid"]\n        if item["error"] or not len(item["images"]):\n            fallback[uid] = item["error"] or "no usable window"\n            return\n        tg = time.perf_counter()\n        try:\n            zmean, zs = score(item["images"], item["meta"])\n        except Exception as exc:\n            fallback[uid] = "gpu: " + repr(exc)[:200]\n            return\n        if not torch.isfinite(zmean).all():\n            fallback[uid] = "non-finite score"\n            return\n        p = torch.sigmoid(zmean).double().cpu().numpy()\n        dt = time.perf_counter() - tg\n        wr.writerow([uid] + ["%.6f" % float(x) for x in p])\n        fh.flush()\n        stats["rows"] += 1\n        stats["decode_s"] += item["seconds"][0]\n        stats["render_s"] += item["seconds"][1]\n        stats["anchor_s"] += item["seconds"][2] if len(item["seconds"]) > 2 else 0.\n        if item.get("side") and item["side"].get("score") is not None:\n            side_counts["checked"] += 1\n            side_counts["flipped"] += int(bool(item["side"].get("flipped")))\n        if item.get("anchor"):\n            anchor_methods[item["anchor"]["method"]] = anchor_methods.get(item["anchor"]["method"], 0) + 1\n        stats["gpu_s"] += dt\n        stats["windows"] += len(item["images"])\n        per_study.append([uid, len(item["images"]), item["seconds"][0], item["seconds"][1], round(dt, 3)])\n        if a.dump_logits:\n            dump[uid] = zs.float().cpu().numpy().tolist()\n\n    pos = 0\n    timed_out = False\n    while pos < len(uids) and not timed_out:\n        ds = CStudies(a.data, a.split, uids[pos:], tmp, a.code, a.size, a.eval_stride, a.views, a.slices, a.anchor_weights, a.phys_mm,\n                      a.side_model, a.side_threshold, a.side_max_x, a.side_position_mm, extent_meta=N_EXTENT[0] > 0)\n        loader = torch.utils.data.DataLoader(ds, batch_size=None, shuffle=False, num_workers=a.workers,\n                                             prefetch_factor=a.prefetch if a.workers else None, persistent_workers=False,\n                                             multiprocessing_context="spawn" if a.workers else None,\n                                             timeout=a.item_timeout if a.workers else 0)\n        try:\n            mark = time.perf_counter()\n            for item in loader:\n                stats["wait_s"] += time.perf_counter() - mark\n                handle(item)\n                pos += 1\n                if pos % 20 == 0 or pos == len(uids) or pos <= 3:\n                    el = time.time() - t_start\n                    print(json.dumps({"done": pos, "of": len(uids), "scored": stats["rows"], "fallback": len(fallback),\n                                      "elapsed_s": round(el, 1), "s_per_study": round(el / max(pos, 1), 2),\n                                      "decode_s": round(stats["decode_s"], 1), "render_s": round(stats["render_s"], 1),\n                                      "gpu_s": round(stats["gpu_s"], 1), "anchor_s": round(stats["anchor_s"], 1),\n                                      "anchors": anchor_methods, "windows": stats["windows"]}), flush=True)\n                if a.deadline and time.time() > a.deadline:\n                    timed_out = True\n                    break\n                mark = time.perf_counter()\n        except Exception as exc:\n            if pos < len(uids):\n                fallback[uids[pos]] = "loader: " + repr(exc)[:200]\n            pos += 1\n        del loader\n    for u in uids[pos:]:\n        fallback[u] = "not reached before the deadline"\n    fh.close()\n    shutil.rmtree(tmp, ignore_errors=True)\n    if a.dump_logits:\n        Path(a.dump_logits).write_text(json.dumps(dump), encoding="utf-8")\n    receipt = {"status": "PASS", "arm": a.arm, "models": [p.name for p in files], "backbone": a.backbone, "shared_trunk": bool(shared_trunk), "init": a.init,\n               "spatial_models": n_spatial, "extent_models": N_EXTENT[0], "group2": None if not models2 else {"models": [p.name for p in files2], "backbone": a.backbone2,\n                                                                                   "spatial_models": n_spatial2, "w2": W2.tolist()},\n               "studies": len(uids), "scored": stats["rows"], "fallback": fallback, "timed_out": timed_out, "amp": amp,\n               "gpu": torch.cuda.get_device_name(), "torch": torch.__version__, "timm": timm.__version__,\n               "seconds": round(time.time() - t_start, 1), "stats": {k: round(v, 1) for k, v in stats.items()},\n               "per_study": per_study[:50], "anchor_methods": anchor_methods, "views": a.views, "phys_mm": a.phys_mm,\n               "side_check": None if not a.side_model else dict(side_counts, threshold=a.side_threshold, max_x=a.side_max_x, position_mm=a.side_position_mm,\n                                                                  model_sha256=hashlib.sha256(Path(a.side_model).read_bytes()).hexdigest()),\n               "anchor_weights_sha256": (hashlib.sha256(Path(a.anchor_weights).read_bytes()).hexdigest() if a.anchor_weights else None),\n               "sha256": {n: hashlib.sha256((HERE / n).read_bytes()).hexdigest()\n                          for n in ("px_cache.py", "compact_train.py", "compact_infer.py", "probe_infer.py")\n                          + (("side_common.py",) if a.side_model else ())\n                          + (("knee_anchor_bank_v2.py",) if a.views in ("v3k", "v4") else ())},\n               "model": "compact fine-tuned image model on the pixel cache"}\n    (out / "receipt.json").write_text(json.dumps(receipt, indent=1), encoding="utf-8")\n    print("WORKER_DONE", json.dumps({k: receipt[k] for k in ("studies", "scored", "timed_out", "seconds", "amp")}),\n          "fallback", len(fallback), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'probe_infer.py': '"""probe_ inference worker (Kaggle or local): raw DICOM -> label-free image scores for one list of studies.\n\nPath, identical to how the probe features were cached:\n  raw DICOM  --mainline training.native_cache.prepare (one study at a time)-->  native uint16 series\n             --fe_extract.StudyWindows.build (the SAME file that built the feature cache)-->  384 px windows\n             --frozen OrthoFoundation-L public SSL weights-->  CLS + 6x6 patch cells --PCA-256 (saved pca.npz)-->\n             --saved probe heads (probe_train.Head, K fold models x seeds), mean logit-->  12 probabilities\nNothing is trained here.  No report, no label, no test-time adaptation.  A study that cannot be decoded, or\nthat is not reached before the deadline, is listed as a fallback (the controller fills it with the mean score).\nEverything here is a probe_ scaffold (frozen encoder + light heads); it is not the mainline model.\n"""\nimport argparse, contextlib, csv, hashlib, inspect, io, json, os, shutil, sys, time\nfrom pathlib import Path\nfrom types import SimpleNamespace\nsys.dont_write_bytecode = True\nHERE = Path(__file__).resolve().parent\nimport numpy as np\nimport torch\n\nUIDK = "StudyInstanceUID"\n\n\nclass OneStudySource:\n    """Directory reader that lists only the study being prepared (the stock one lists every DICOM of the split)."""\n    current = None\n\n    def __init__(self, root, split):\n        self.root = Path(root)\n        self.split = split\n\n    def read(self, item):\n        return (self.root / (item.filename if hasattr(item, "filename") else item)).read_bytes()\n\n    def infolist(self):\n        base = self.root / (self.split + "_series") / OneStudySource.current\n        return [SimpleNamespace(filename=p.relative_to(self.root).as_posix()) for p in sorted(base.glob("*/*.dcm"))]\n\n    def close(self):\n        pass\n\n\ndef quantize_series_slicewise(images):\n    """Same global float32 extrema and per-voxel arithmetic as training.native_cache.quantize_series, without\n    the whole-volume float32 copy (taken from the mainline\'s accepted Kaggle memory patch)."""\n    lo = min(float(np.asarray(x, np.float32).min()) for x in images)\n    hi = max(float(np.asarray(x, np.float32).max()) for x in images)\n    scale = max(hi - lo, 1e-6)\n    encoded = np.empty((len(images), *images[0].shape), dtype=np.uint16)\n    for z, image in enumerate(images):\n        volume = np.asarray(image, np.float32)\n        encoded[z] = np.rint((volume - lo) / scale * 65535).clip(0, 65535).astype(np.uint16)\n    window = np.percentile(encoded[:, ::4, ::4], [1, 99]).astype(float).tolist()\n    return encoded, {"minimum": lo, "maximum": hi, "max_absolute_quantization_error_bound": scale / 131070,\n                     "display_window_uint16": window}\n\n\ndef install_native(lean=True):\n    import training.native_cache as native\n    if lean:\n        src = inspect.getsource(native.prepare)\n        for before, after in [\n            ("pydicom.dcmread(io.BytesIO(archive.read(i)))",\n             "pydicom.dcmread(io.BytesIO(archive.read(i)), stop_before_pixels=True)"),\n            ("geom.pixels_from_dataset(objects[sid][name],path=name)",\n             "geom.pixels_from_dataset(pydicom.dcmread(io.BytesIO(archive.read(name))),path=name)")]:\n            if src.count(before) != 1:\n                raise RuntimeError("training.native_cache.prepare differs from the frozen source")\n            src = src.replace(before, after)\n        exec(compile(src, "probe_infer:prepare", "exec"), native.__dict__)\n        native.quantize_series = quantize_series_slicewise\n    native.DirectorySource = OneStudySource\n    return native\n\n\nclass Studies(torch.utils.data.Dataset):\n    def __init__(self, data, split, uids, tmp, code, lean=True):\n        self.data, self.split, self.uids = str(data), split, list(uids)\n        self.tmp, self.code, self.lean = str(tmp), str(code), lean\n        self.native = None\n        self.windows = None\n\n    def __len__(self):\n        return len(self.uids)\n\n    def _setup(self):\n        if self.code not in sys.path:\n            sys.path.insert(0, self.code)\n        if str(HERE) not in sys.path:\n            sys.path.insert(0, str(HERE))\n        self.native = install_native(self.lean)\n        import fe_extract\n        self.windows = fe_extract.StudyWindows(self.tmp, [])\n\n    def __getitem__(self, i):\n        uid = self.uids[i]\n        t0 = time.perf_counter()\n        work = Path(self.tmp) / ("s_" + hashlib.sha256(uid.encode()).hexdigest()[:16])\n        t1 = t0\n        try:\n            if self.native is None:\n                self._setup()\n            OneStudySource.current = uid\n            with contextlib.redirect_stdout(io.StringIO()):\n                self.native.prepare(self.data, work, split=self.split, study_ids=[uid], source_directory=True)\n            t1 = time.perf_counter()\n            self.windows.cache = str(work)\n            item = self.windows.build(uid)\n            item["error"] = None\n        except Exception as exc:\n            item = {"uid": uid, "error": repr(exc)[:300], "images": torch.zeros((0, 3, 384, 384), dtype=torch.uint8),\n                    "meta": torch.zeros((0, 12), dtype=torch.float32), "series": [], "removed": 0}\n        finally:\n            shutil.rmtree(work, ignore_errors=True)\n        item["seconds"] = [round(t1 - t0, 3), round(time.perf_counter() - t1, 3)]\n        return item\n\n\ndef load_heads(heads_dir, arm, hidden, dev):\n    if str(HERE) not in sys.path:\n        sys.path.insert(0, str(HERE))\n    from probe_train import Head, LABELS\n    if Path(heads_dir).is_file():                     # one file: {"<arm>_s<seed>_<fold>.pt": state_dict}\n        blob = torch.load(heads_dir, map_location="cpu", weights_only=True)\n        items = sorted((k, v) for k, v in blob.items() if k.startswith(arm + "_s"))\n    else:\n        items = [(p.name, torch.load(p, map_location="cpu", weights_only=True))\n                 for p in sorted(Path(heads_dir).glob(arm + "_s*_*.pt"))]\n    if not items:\n        raise SystemExit("no head files for arm " + arm)\n    heads = []\n    for name, state in items:\n        h = Head(hidden)\n        h.load_state_dict(state, strict=True)\n        heads.append(h.eval().requires_grad_(False).to(dev))\n    return heads, LABELS, [name for name, _ in items]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True, help="folder with <split>.csv, <split>_series.csv, <split>_series/")\n    ap.add_argument("--split", default="test")\n    ap.add_argument("--code", required=True, help="folder that contains the mainline \'training\' package")\n    ap.add_argument("--weights", required=True)\n    ap.add_argument("--pca", required=True)\n    ap.add_argument("--norm", required=True)\n    ap.add_argument("--heads", required=True)\n    ap.add_argument("--arm", required=True)\n    ap.add_argument("--hidden", type=int, default=256)\n    ap.add_argument("--assignment", default=None, help="json list of study ids; default = every id of <split>.csv")\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--tmp", default=None)\n    ap.add_argument("--amp", default="auto", choices=["auto", "fp16", "bf16", "fp32"])\n    ap.add_argument("--batch", type=int, default=64)\n    ap.add_argument("--workers", type=int, default=2)\n    ap.add_argument("--prefetch", type=int, default=1)\n    ap.add_argument("--item-timeout", type=float, default=900., help="seconds to wait for one study from the loader")\n    ap.add_argument("--deadline", type=float, default=0., help="unix time after which no new study is started")\n    ap.add_argument("--dump-features", default=None)\n    ap.add_argument("--stock-native", action="store_true", help="use the unpatched native builder (parity checks)")\n    a = ap.parse_args()\n    out = Path(a.output)\n    out.mkdir(parents=True, exist_ok=True)\n    tmp = Path(a.tmp or (out / "tmp"))\n    tmp.mkdir(parents=True, exist_ok=True)\n    t_start = time.time()\n    if a.assignment:\n        uids = json.loads(Path(a.assignment).read_text(encoding="utf-8"))\n    else:\n        with open(Path(a.data) / (a.split + ".csv"), encoding="utf-8") as fh:\n            uids = [r[UIDK] for r in csv.DictReader(fh)]\n    if len(uids) != len(set(uids)):\n        raise SystemExit("duplicate study ids")\n\n    dev = torch.device("cuda")\n    if a.code not in sys.path:\n        sys.path.insert(0, a.code)\n    if str(HERE) not in sys.path:\n        sys.path.insert(0, str(HERE))\n    import fe_extract as FE\n    import timm\n    from safetensors.torch import load_file\n    cap = torch.cuda.get_device_capability()[0]\n    amp = a.amp if a.amp != "auto" else ("bf16" if cap >= 8 else "fp16")\n    torch.backends.cuda.matmul.allow_tf32 = True\n    torch.backends.cudnn.allow_tf32 = True\n    model = timm.create_model("vit_large_patch16_dinov3_qkvb", pretrained=False, num_classes=0, global_pool="",\n                              img_size=FE.IMG)\n    model.load_state_dict(load_file(a.weights), strict=True)\n    model.rope = None\n    model.eval().requires_grad_(False).to(dev)\n    prefix = int(getattr(model, "num_prefix_tokens", 5))\n    mean = torch.tensor([.485, .456, .406], device=dev).view(1, 3, 1, 1)\n    std = torch.tensor([.229, .224, .225], device=dev).view(1, 3, 1, 1)\n    g = FE.IMG // 16\n    blk = g // FE.GRID\n    z = np.load(a.pca)\n    pca_mean, pca_comp = torch.from_numpy(z["mean"]).to(dev), torch.from_numpy(z["components"]).to(dev)\n    zn = np.load(a.norm)\n    c_mu, c_sd = torch.from_numpy(zn["mu"]).to(dev), torch.from_numpy(zn["sd"]).to(dev)\n    heads, LABELS, head_names = load_heads(a.heads, a.arm, a.hidden, dev)\n    print("MODEL_READY", json.dumps({"gpu": torch.cuda.get_device_name(), "amp": amp, "heads": len(heads), "studies": len(uids), "seconds": round(time.time() - t_start, 1)}), flush=True)\n    retries = {"fp32_batches": 0}\n\n    def forward_tokens(x, mode):\n        if mode == "fp32":\n            return model.forward_features(x).float()\n        dt = torch.float16 if mode == "fp16" else torch.bfloat16\n        with torch.autocast("cuda", dtype=dt):\n            return model.forward_features(x).float()\n\n    @torch.inference_mode()\n    def encode(images_u8):\n        out_cls, out_cells = [], []\n        for chunk in images_u8.split(a.batch):\n            x = chunk.to(dev, non_blocking=True).float().div_(255.)\n            x = (x - mean) / std\n            tok = forward_tokens(x, amp)\n            if not torch.isfinite(tok).all():\n                tok = forward_tokens(x, "fp32")\n                retries["fp32_batches"] += 1\n            b = tok.shape[0]\n            if tok.shape[1] != prefix + g * g:\n                raise ValueError("unexpected token count %s" % (tuple(tok.shape),))\n            patch = tok[:, prefix:].reshape(b, FE.GRID, blk, FE.GRID, blk, tok.shape[-1]).mean((2, 4))\n            out_cls.append(tok[:, 0])\n            out_cells.append(patch.reshape(b, FE.GRID * FE.GRID, tok.shape[-1]))\n        return torch.cat(out_cls), torch.cat(out_cells)\n\n    @torch.inference_mode()\n    def score(cls, cells, meta):\n        cls16 = cls.half()                                             # the cache stores float16\n        cells16 = ((cells - pca_mean) @ pca_comp).half()\n        x_cells = ((cells16.float() - c_mu) / c_sd).clamp_(-8, 8).half().float()\n        x_cls = cls16.float()\n        meta = meta.to(dev)\n        slot = meta[:, 1].long().clamp(0, 9)[None]\n        view = meta[:, 3].long().clamp(0, 5)[None]\n        geom = meta[:, 4:8][None]\n        mask = torch.ones((1, len(cls)), dtype=torch.bool, device=dev)\n        zs = torch.stack([h(x_cls[None], x_cells[None], slot, view, geom, mask)[0] for h in heads])\n        return zs.mean(0), zs, cls16, cells16\n\n    pred_path = out / "predictions.csv"\n    fh = open(pred_path, "w", encoding="utf-8", newline="")\n    wr = csv.writer(fh)\n    wr.writerow([UIDK] + LABELS)\n    fallback, stats = {}, {"decode_s": 0., "render_s": 0., "gpu_s": 0., "windows": 0, "wait_s": 0., "rows": 0}\n    per_study = []\n    if a.dump_features:\n        Path(a.dump_features).mkdir(parents=True, exist_ok=True)\n\n    def handle(item):\n        uid = item["uid"]\n        if item["error"] or not len(item["images"]):\n            fallback[uid] = item["error"] or "no usable window"\n            return\n        tg = time.perf_counter()\n        try:\n            cls, cells = encode(item["images"])\n            zmean, zs, cls16, cells16 = score(cls, cells, item["meta"])\n        except Exception as exc:\n            fallback[uid] = "gpu: " + repr(exc)[:200]\n            return\n        if not torch.isfinite(zmean).all():\n            fallback[uid] = "non-finite score"\n            return\n        p = torch.sigmoid(zmean).double().cpu().numpy()\n        dt = time.perf_counter() - tg\n        wr.writerow([uid] + ["%.6f" % float(x) for x in p])\n        fh.flush()\n        stats["rows"] += 1\n        stats["decode_s"] += item["seconds"][0]\n        stats["render_s"] += item["seconds"][1]\n        stats["gpu_s"] += dt\n        stats["windows"] += len(cls)\n        per_study.append([uid, len(cls), item["seconds"][0], item["seconds"][1], round(dt, 3)])\n        if a.dump_features:\n            np.savez(os.path.join(a.dump_features, hashlib.sha256(uid.encode()).hexdigest()[:24] + ".npz"), uid=uid,\n                     cls=cls16.cpu().numpy(), cells=cells16.cpu().numpy(), meta=item["meta"].numpy(),\n                     logits=zs.float().cpu().numpy(), series=np.array(item["series"]))\n\n    pos = 0\n    timed_out = False\n    while pos < len(uids) and not timed_out:\n        ds = Studies(a.data, a.split, uids[pos:], tmp, a.code, lean=not a.stock_native)\n        # spawn: the same start method as the Windows runs this was tested with (no fork-after-CUDA state);\n        # timeout: a stuck decode must surface as an error for that study, never as a silent hang\n        loader = torch.utils.data.DataLoader(ds, batch_size=None, shuffle=False, num_workers=a.workers,\n                                             prefetch_factor=a.prefetch if a.workers else None, persistent_workers=False,\n                                             multiprocessing_context="spawn" if a.workers else None,\n                                             timeout=a.item_timeout if a.workers else 0)\n        try:\n            mark = time.perf_counter()\n            for item in loader:\n                stats["wait_s"] += time.perf_counter() - mark\n                handle(item)\n                pos += 1\n                if pos % 20 == 0 or pos == len(uids) or pos <= 3:\n                    el = time.time() - t_start\n                    print(json.dumps({"done": pos, "of": len(uids), "scored": stats["rows"], "fallback": len(fallback),\n                                      "elapsed_s": round(el, 1), "s_per_study": round(el / max(pos, 1), 2),\n                                      "decode_s": round(stats["decode_s"], 1), "render_s": round(stats["render_s"], 1),\n                                      "gpu_s": round(stats["gpu_s"], 1), "windows": stats["windows"]}), flush=True)\n                if a.deadline and time.time() > a.deadline:\n                    timed_out = True\n                    break\n                mark = time.perf_counter()\n        except Exception as exc:                       # a loader worker died: skip that study, restart after it\n            if pos < len(uids):\n                fallback[uids[pos]] = "loader: " + repr(exc)[:200]\n            pos += 1\n        del loader\n    for u in uids[pos:]:\n        fallback[u] = "not reached before the deadline"\n    fh.close()\n    shutil.rmtree(tmp, ignore_errors=True)\n    receipt = {"status": "PASS", "arm": a.arm, "heads": head_names, "studies": len(uids), "scored": stats["rows"],\n               "fallback": fallback, "timed_out": timed_out, "amp": amp, "fp32_retry_batches": retries["fp32_batches"],\n               "gpu": torch.cuda.get_device_name(), "torch": torch.__version__, "timm": timm.__version__,\n               "seconds": round(time.time() - t_start, 1), "stats": {k: round(v, 1) for k, v in stats.items()},\n               "per_study": per_study[:50],\n               "native": "stock" if a.stock_native else "lean (header pass, slice-wise quantise)",\n               "sha256": {n: hashlib.sha256((HERE / n).read_bytes()).hexdigest()\n                          for n in ("fe_extract.py", "probe_train.py", "probe_infer.py")},\n               "scaffold": "probe_: frozen encoder + light heads"}\n    (out / "receipt.json").write_text(json.dumps(receipt, indent=1), encoding="utf-8")\n    print("WORKER_DONE", json.dumps({k: receipt[k] for k in ("studies", "scored", "timed_out", "seconds", "amp")}),\n          "fallback", len(fallback), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'probe_dual.py': '"""probe_ controller: split the test studies over the visible GPUs, run probe_infer.py workers, merge.\n\nNever fails the notebook for a single bad study or a dead worker: whatever a worker wrote is used, every\nother study gets the per-class mean of the scored studies (0.5 if nothing was scored) and is counted in the\nreceipt.  Output columns and row order follow sample_submission.csv.\n"""\nimport argparse, csv, json, os, subprocess, sys, time\nfrom pathlib import Path\nsys.dont_write_bytecode = True\nHERE = Path(__file__).resolve().parent\nUIDK = "StudyInstanceUID"\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--split", default="test")\n    ap.add_argument("--code", required=True)\n    ap.add_argument("--bundle", required=True, help="folder with model.safetensors, pca.npz, feature_norm.npz, heads/")\n    ap.add_argument("--arm", required=True)\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--amp", default="auto")\n    ap.add_argument("--loader-workers", type=int, default=2)\n    ap.add_argument("--deadline", type=float, default=0.)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--weights", default=None, help="default: <bundle>/model.safetensors")\n    ap.add_argument("--worker", default="probe_infer.py", help="probe_infer.py or compact_infer.py")\n    ap.add_argument("--worker-args", default="[]", help="json list of extra worker arguments (compact worker)")\n    ap.add_argument("--max-fallback-fraction", type=float, default=0.03)\n    a = ap.parse_args()\n    t0 = time.time()\n    data, out, bundle = Path(a.data), Path(a.output), Path(a.bundle)\n    out.mkdir(parents=True, exist_ok=True)\n    with open(data / (a.split + ".csv"), encoding="utf-8") as fh:\n        ids = [r[UIDK] for r in csv.DictReader(fh)]\n    if a.limit:\n        ids = ids[:a.limit]\n    if len(ids) != len(set(ids)):\n        raise SystemExit("duplicate test ids")\n    try:\n        names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],\n                                        text=True).strip().splitlines()\n    except Exception:\n        names = ["unknown"]\n    n_gpu = max(1, len(names))\n\n    def cost(uid):                       # number of DICOM files; scheduling only\n        n = 0\n        try:\n            for s in os.scandir(data / (a.split + "_series") / uid):\n                if s.is_dir():\n                    n += sum(1 for _ in os.scandir(s.path))\n        except OSError:\n            pass\n        return n\n\n    costs = {u: cost(u) for u in ids}\n    order = sorted(ids, key=lambda u: (-costs[u], u))\n    parts = [order[i::n_gpu] for i in range(n_gpu)]\n    # inside a worker: keep the big studies from arriving back to back\n    parts = [p[::2] + p[1::2] for p in parts]\n    heads_path = str(bundle / "heads.pt") if (bundle / "heads.pt").is_file() else str(bundle / "heads")\n    procs = []\n    for gi, part in enumerate(parts):\n        wdir = out / ("worker%d" % gi)\n        wdir.mkdir(exist_ok=True)\n        assign = wdir / "ids.json"\n        assign.write_text(json.dumps(part), encoding="utf-8")\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gi), OMP_NUM_THREADS="1", MKL_NUM_THREADS="1",\n                   OPENBLAS_NUM_THREADS="1", MALLOC_ARENA_MAX="2", PYTHONDONTWRITEBYTECODE="1")\n        cmd = [sys.executable, "-u", str(HERE / a.worker), "--data", str(data), "--split", a.split,\n               "--code", a.code, "--arm", a.arm, "--assignment", str(assign), "--output", str(wdir), "--amp", a.amp,\n               "--workers", str(a.loader_workers), "--deadline", str(a.deadline)]\n        if a.worker == "probe_infer.py":\n            cmd += ["--weights", a.weights or str(bundle / "model.safetensors"), "--pca", str(bundle / "pca.npz"),\n                    "--norm", str(bundle / "feature_norm.npz"), "--heads", heads_path]\n        else:\n            cmd += ["--models", str(bundle)] + [str(x) for x in json.loads(a.worker_args)]\n        log = open(wdir / "log.txt", "w", encoding="utf-8")\n        procs.append((gi, subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log, wdir, part))\n        print("WORKER_START", gi, "studies", len(part), "dicom files", sum(costs[u] for u in part), flush=True)\n    last = 0\n    while any(p.poll() is None for _, p, _, _, _ in procs):\n        time.sleep(5)\n        if time.time() - last > 60:\n            last = time.time()\n            for gi, p, _, wdir, part in procs:\n                try:\n                    tail = [l for l in (wdir / "log.txt").read_text(encoding="utf-8", errors="replace").splitlines()\n                            if l.startswith("{")][-1:]\n                except OSError:\n                    tail = []\n                print("PROGRESS worker", gi, "of", len(part), tail[0] if tail else "(starting)", flush=True)\n    codes = {}\n    for gi, p, log, wdir, part in procs:\n        log.close()\n        codes[gi] = p.returncode\n        if p.returncode != 0:\n            print("WORKER_FAILED", gi, "exit", p.returncode, flush=True)\n            print((wdir / "log.txt").read_text(encoding="utf-8", errors="replace")[-3000:], flush=True)\n\n    with open(data / "sample_submission.csv", encoding="utf-8") as fh:\n        rd = csv.reader(fh)\n        header = next(rd)\n        sample_ids = [r[0] for r in rd]\n    if a.limit:\n        sample_ids = [u for u in sample_ids if u in set(ids)]\n    if set(sample_ids) != set(ids):\n        raise SystemExit("test.csv and sample_submission.csv list different studies")\n    labels = header[1:]\n    pred, receipts = {}, []\n    for gi, p, log, wdir, part in procs:\n        f = wdir / "predictions.csv"\n        if f.exists():\n            with open(f, encoding="utf-8") as fh:\n                for r in csv.DictReader(fh):\n                    try:\n                        v = [float(r[L]) for L in labels]\n                    except (KeyError, ValueError, TypeError):\n                        continue\n                    if all(0. <= x <= 1. for x in v):\n                        pred[r[UIDK]] = v\n        try:\n            receipts.append(json.loads((wdir / "receipt.json").read_text(encoding="utf-8")))\n        except Exception:\n            receipts.append({"status": "NO_RECEIPT", "worker": gi})\n    scored = [u for u in sample_ids if u in pred]\n    if scored:\n        mean = [sum(pred[u][j] for u in scored) / len(scored) for j in range(len(labels))]\n    else:\n        mean = [0.5] * len(labels)\n    missing = [u for u in sample_ids if u not in pred]\n    with open(out / "submission.csv", "w", encoding="utf-8", newline="") as fh:\n        wr = csv.writer(fh)\n        wr.writerow(header)\n        for u in sample_ids:\n            wr.writerow([u] + ["%.6f" % x for x in pred.get(u, mean)])\n    reasons = {}\n    for r in receipts:\n        for u, why in (r.get("fallback") or {}).items():\n            reasons[u] = why\n    total = {"decode_s": 0., "render_s": 0., "gpu_s": 0., "windows": 0., "rows": 0.}\n    for r in receipts:\n        for k in total:\n            total[k] += float((r.get("stats") or {}).get(k, 0.))\n    n = max(1., total["rows"])\n    result = {"status": "PASS" if scored else "ALL_FALLBACK", "arm": a.arm, "rows": len(sample_ids),\n              "scored": len(scored), "fallback_studies": len(missing),\n              "fallback_reasons": {u: reasons.get(u, "worker did not report") for u in missing[:40]},\n              "worker_exit_codes": codes, "gpus": names, "seconds": round(time.time() - t0, 1),\n              "per_study_mean": {"decode_s": round(total["decode_s"] / n, 3), "render_s": round(total["render_s"] / n, 3),\n                                 "gpu_s": round(total["gpu_s"] / n, 3), "windows": round(total["windows"] / n, 1)},\n              "workers": receipts, "worker": a.worker}\n    (out / "submission_receipt.json").write_text(json.dumps(result, indent=1), encoding="utf-8")\n    if len(missing) > a.max_fallback_fraction * len(sample_ids):\n        raise SystemExit("PROBE_TOO_MANY_FALLBACKS %d of %d (limit %.0f%%); see submission_receipt.json" % (\n            len(missing), len(sample_ids), 100 * a.max_fallback_fraction))\n    print("CONTROLLER_DONE", json.dumps({k: result[k] for k in ("status", "rows", "scored", "fallback_studies",\n                                                                "seconds", "per_study_mean", "worker_exit_codes")}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'knee_anchor_bank_v2.py': '"""Knee anchor bank v2: the knee joint centre (and patella centre) of every study in world coordinates, projected into\neach series\' pixel grid.  Library (AnchorBank, used by compact_infer.py at Kaggle inference) + CLI (the 5090 bank).\n\nPer study: every aligned sagittal series yields a candidate = centroid of the SKM-TEA joint tissues (any of femoral /\ntibial cartilage / meniscus confidently present: max over the three channels > 0.5) on 7 slices ~4 mm apart around the\nmiddle of the stack.  A candidate is kept when its mass >= MASS_MIN pixels, it projects inside the central 15-85 % of\nits own series\' rows and columns, and it lies within PRIOR_MM of the slab prior.  Candidates within AGREE_MM are\nmass-averaged; otherwise the heaviest wins (\'disagree\' recorded).  With no candidate the anchor is the slab prior.\nSlab prior = centre of the intersection of the orthogonal stacks: LR from the sagittal stack, AP from the coronal stack,\nSI from the sagittal+coronal field centres (header geometry only, no model).\nFixes over v1 (found on the 58 gold montages): v1\'s \'sum of three probabilities > 0.5\' mask fired on calf tissue in\nlarge-FOV scans (joint at row 0.91-0.96 accepted); v1 took only the first fluid-sensitive series; v1 rejected\noblique-sagittal protocols (cos < 0.97) wholesale; v1\'s fallback was the sagittal field centre.\nNon-fluid-sensitive sagittal series (T1 / PD without fat suppression) give ~zero mass: expected, not a flip error\n(checked on 136 gold series x 4 flip combinations).\n\nCLI (5090):  python knee_anchor_bank_v2.py --out F:/rsnaknee/anchors_v2 [--uids list.json] [--limit N]\nOutput anchors/<sha24(study)>.json: {"study", "version": 2, "joint_lps", "patella_lps", "slab_prior_lps", "method",\n  "why": [...], "series": {sid: {"joint": [row_frac, col_frac], "patella": [...] | null, "in_field", "dist_mm"}}}\nNo label is read anywhere here."""\nimport argparse, csv, hashlib, json, os, sys, time\nimport numpy as np, cv2, torch, torch.nn.functional as F\n\nGEO = "F:/rsnaknee/geometry"            # 5090 defaults (CLI); the library takes explicit directories\nCACHE = "F:/rsnaknee/cache_native"\nLO, HI = 0.15, 0.85          # plausibility window (fractions of the source series\' rows / cols)\nAGREE_MM = 20.0\nPRIOR_MM = 35.0              # a segmentation candidate farther than this from the slab prior is rejected (g35: 85 mm)\nOBLIQUE_COS = 0.85           # v1 used 0.97 (14 deg): oblique-sagittal knee protocols were rejected wholesale\nSPAN_MM = 4.0                # spacing between the segmented slices (3-D stacks otherwise contribute a 3 mm slab)\nMASS_MIN = 250               # v1: 400\nN_SLICES = 7\n\n\ndef key_for(uid, sid):\n    return hashlib.sha256((uid + "/" + sid).encode()).hexdigest()[:24]\n\n\ndef study_key(uid):\n    return hashlib.sha256(uid.encode()).hexdigest()[:24]\n\n\ndef load_model(weights, dev):\n    """the SKM-TEA V-Net through the mainline loader (needs the `training` package on sys.path)."""\n    from training.anatomy import load_segmenter\n    return load_segmenter(weights, dev)\n\n\ndef project(aff, X):\n    """aff [Z,3,3] (col,row,1 -> LPS mm); X [3] LPS -> (z, col, row, |distance to that plane|) of the nearest slice."""\n    o = aff[:, :, 2]\n    n = np.cross(aff[0, :, 0], aff[0, :, 1]); n /= np.linalg.norm(n)\n    d = (o - X[None]) @ n\n    z = int(np.argmin(np.abs(d)))\n    M = aff[z, :, :2]\n    cr, *_ = np.linalg.lstsq(M, X - o[z], rcond=None)\n    return z, float(cr[0]), float(cr[1]), float(abs(d[z]))\n\n\ndef centroid(prob, thr=0.5):\n    m = prob > thr\n    if m.sum() < 40:\n        return None, 0\n    ys, xs = np.nonzero(m)\n    w = prob[m]\n    return (float((ys * w).sum() / w.sum()), float((xs * w).sum() / w.sum())), int(m.sum())\n\n\nclass AnchorBank:\n    def __init__(self, model, dev, geo_dir=None):\n        self.model, self.dev, self.geo_dir = model, dev, geo_dir\n\n    def geometry(self, uid, sid, geo_dir=None):\n        p = os.path.join(geo_dir or self.geo_dir, key_for(uid, sid) + ".json")\n        return json.load(open(p)) if os.path.exists(p) else None\n\n    @torch.no_grad()\n    def segment(self, series, geo, n_slices=N_SLICES):\n        from training.anatomy_runtime import native_volume\n        vol = native_volume(series["path"])\n        mean, std = float(vol.mean()), max(float(vol.std()), 1.)\n        aff = np.asarray(geo["affines_col_row_origin_lps_mm"], float)\n        if abs(aff[0, 1, 0]) / np.linalg.norm(aff[0, :, 0]) < OBLIQUE_COS or abs(aff[0, 2, 1]) / np.linalg.norm(aff[0, :, 1]) < OBLIQUE_COS:\n            return None, "oblique"\n        flip_x = aff[0, 1, 0] < 0; flip_y = aff[0, 2, 1] > 0          # same convention as training.anatomy.segment_series\n        m = len(vol) // 2\n        dz = float(np.linalg.norm(aff[1, :, 2] - aff[0, :, 2])) if len(aff) > 1 else 4.\n        step = max(1, int(round(SPAN_MM / max(dz, 1e-3))))\n        zs = sorted({min(len(vol) - 1, max(0, m + k * step)) for k in range(-(n_slices // 2), n_slices // 2 + 1)})\n        out = []\n        for z in zs:\n            x = (np.asarray(vol[z], np.float32) - mean) / std\n            if flip_x: x = x[:, ::-1]\n            if flip_y: x = x[::-1]\n            sy, sx = series["spacing_mm"][z]\n            shape = (max(16, round(x.shape[1] * sx / .3125)), max(16, round(x.shape[0] * sy / .3125)))\n            x = cv2.resize(np.ascontiguousarray(x), shape, interpolation=cv2.INTER_LINEAR)\n            t = torch.from_numpy(x.copy())[None, None].to(self.dev)\n            h, w = t.shape[-2:]; t = F.pad(t, (0, (-w) % 16, 0, (-h) % 16), mode="replicate")\n            p = self.model(t).sigmoid()[..., :h, :w]\n            p = F.interpolate(p, size=vol.shape[-2:], mode="bilinear", align_corners=False)[0].float().cpu().numpy()\n            if flip_x: p = p[:, :, ::-1]\n            if flip_y: p = p[:, ::-1]\n            out.append((z, np.ascontiguousarray(p)))\n        return out, "ok"\n\n    def candidate(self, s, geo):\n        aff = np.asarray(geo["affines_col_row_origin_lps_mm"], float)\n        segs, why = self.segment(s, geo)\n        if segs is None:\n            return None, why\n        pts, wts, ppts, pwts = [], [], [], []\n        for z, p in segs:\n            joint = np.max(p[1:4], axis=0)                       # any joint tissue confidently present\n            c, n = centroid(joint, 0.5)\n            if c is not None:\n                pts.append(aff[z] @ np.array([c[1], c[0], 1.])); wts.append(n)\n            cp, npx = centroid(p[0], 0.5)\n            if cp is not None and npx >= 60:\n                ppts.append(aff[z] @ np.array([cp[1], cp[0], 1.])); pwts.append(npx)\n        mass, pmass = int(sum(wts)), int(sum(pwts))\n        if mass < MASS_MIN:\n            return None, "mass<%d(%d)" % (MASS_MIN, mass)\n        X = np.average(np.array(pts), axis=0, weights=np.array(wts, float))\n        Xp = np.average(np.array(ppts), axis=0, weights=np.array(pwts, float)) if pmass >= 120 else None\n        if Xp is not None and not (Xp[1] < X[1] and Xp[2] > X[2] - 10):   # patella: anterior (smaller LPS y), not below the joint\n            Xp = None\n        h, w = s["native_shape"][-2:]\n        z, c, r, _ = project(aff, X)\n        rf, cf = r / max(1, h - 1), c / max(1, w - 1)\n        if not (LO <= rf <= HI and LO <= cf <= HI):\n            return None, "implausible(rf=%.2f,cf=%.2f)" % (rf, cf)\n        return dict(X=X, Xp=Xp, mass=mass, pmass=pmass, sid=s["SeriesInstanceUID"]), "ok"\n\n    def slab_prior(self, uid, d, geo_dir=None):\n        per = {}\n        for s in d["series"]:\n            geo = self.geometry(uid, s["SeriesInstanceUID"], geo_dir)\n            if geo is None or not geo["quality"]["aligned"]:\n                continue\n            aff = np.asarray(geo["affines_col_row_origin_lps_mm"], float)\n            h, w = s["native_shape"][-2:]; z = len(aff) // 2\n            per.setdefault(s["plane"], []).append(aff[z] @ np.array([(w - 1) / 2., (h - 1) / 2., 1.]))\n        if not per:\n            return None\n        G = np.zeros(3)\n        for ax, planes in enumerate((("Sagittal",), ("Coronal",), ("Sagittal", "Coronal"))):\n            cs = [c for pl in planes for c in per.get(pl, [])] or [c for pl in per for c in per[pl]]\n            G[ax] = float(np.median([c[ax] for c in cs]))\n        return G\n\n    def record(self, uid, d, geo_dir=None):\n        """anchor record of one study from its native-cache study dict `d` and the geometry directory."""\n        sags = [s for s in d["series"] if s["plane"] == "Sagittal" and os.path.exists(s["path"])]\n        sags.sort(key=lambda s: -(s.get("fluid_sensitive") or 0))\n        rec = {"study": uid, "version": 2, "joint_lps": None, "patella_lps": None, "method": None, "why": [], "series": {}}\n        G = self.slab_prior(uid, d, geo_dir)\n        rec["slab_prior_lps"] = None if G is None else [float(v) for v in G]\n        cands = []\n        for s in sags:\n            geo = self.geometry(uid, s["SeriesInstanceUID"], geo_dir)\n            if geo is None or not geo["quality"]["aligned"]:\n                rec["why"].append("no_geo" if geo is None else "not_aligned"); continue\n            try:\n                c, why = self.candidate(s, geo)\n            except Exception as exc:                                   # one bad series must not lose the study\n                c, why = None, "error:" + repr(exc)[:80]\n            if c is None:\n                rec["why"].append(why); continue\n            if G is not None and np.linalg.norm(c["X"] - G) > PRIOR_MM:\n                rec["why"].append("far_from_prior(%.0fmm)" % np.linalg.norm(c["X"] - G)); continue\n            cands.append(c)\n        X = Xp = None\n        if cands:\n            cands.sort(key=lambda c: -c["mass"])\n            best = cands[0]\n            close = [c for c in cands if np.linalg.norm(c["X"] - best["X"]) <= AGREE_MM]\n            if len(close) >= 2:\n                X = np.average(np.array([c["X"] for c in close]), axis=0, weights=np.array([c["mass"] for c in close], float))\n                rec["method"] = "skm_tea_joint_avg%d" % len(close)\n            else:\n                X = best["X"]; rec["method"] = "skm_tea_joint"\n                if len(cands) >= 2:\n                    rec["why"].append("disagree(%.0fmm)" % np.linalg.norm(cands[1]["X"] - best["X"]))\n            pc = [c for c in close if c["Xp"] is not None] or [c for c in cands if c["Xp"] is not None]\n            if pc:\n                Xp = max(pc, key=lambda c: c["pmass"])["Xp"]\n        elif G is not None:\n            X = G; rec["method"] = "slab_prior"\n        else:\n            rec["why"].append("no_geometry")\n        if X is not None:\n            rec["joint_lps"] = [float(v) for v in X]\n            if Xp is not None:\n                rec["patella_lps"] = [float(v) for v in Xp]\n            for s in d["series"]:\n                geo = self.geometry(uid, s["SeriesInstanceUID"], geo_dir)\n                if geo is None:\n                    continue\n                aff = np.asarray(geo["affines_col_row_origin_lps_mm"], float)\n                h, w = s["native_shape"][-2:]\n                z, c, r, dist = project(aff, X)\n                entry = {"joint": [r / max(1, h - 1), c / max(1, w - 1)], "in_field": bool(0 <= r < h and 0 <= c < w),\n                         "dist_mm": round(dist, 1), "patella": None}\n                if Xp is not None:\n                    zp, cp, rp, dp = project(aff, Xp)\n                    entry["patella"] = [rp / max(1, h - 1), cp / max(1, w - 1)]\n                rec["series"][s["SeriesInstanceUID"]] = entry\n        return rec\n\n\ndef plausible_v1(rec, d):\n    """does an existing record pass the v2 gate?  (joint fractions of every sagittal series inside the window)"""\n    if not str(rec.get("method", "")).startswith("skm_tea_joint"):\n        return False\n    for s in d["series"]:\n        if s["plane"] != "Sagittal":\n            continue\n        e = rec["series"].get(s["SeriesInstanceUID"])\n        if e is None or not e["in_field"]:\n            return False\n        rf, cf = e["joint"]\n        if not (LO <= rf <= HI and LO <= cf <= HI):\n            return False\n    return True\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--out", default="F:/rsnaknee/anchors_v2")\n    ap.add_argument("--geo", default=GEO)\n    ap.add_argument("--cache", default=CACHE)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--uids", default=None)\n    ap.add_argument("--only-rework", default=None, help="existing anchor dir: process only fallback / implausible records")\n    ap.add_argument("--weights", default="F:/rsnaknee/pretrained/skm_tea/model-raw.ckpt")\n    ap.add_argument("--code-root", default="D:/rsnaknee", help="directory holding the mainline `training` package")\n    a = ap.parse_args()\n    if a.code_root and os.path.isdir(a.code_root) and a.code_root not in sys.path:\n        sys.path.insert(0, a.code_root)\n    os.makedirs(a.out, exist_ok=True)\n    dev = "cuda"\n    bank = AnchorBank(load_model(a.weights, dev), dev, a.geo)\n    uids = [r["StudyInstanceUID"] for r in csv.DictReader(open(os.path.join(a.cache, "patients.csv")))]\n    if a.uids: uids = json.load(open(a.uids))\n    if a.limit: uids = uids[:a.limit]\n    t0 = time.time(); stats = {"ok": 0, "ok_avg": 0, "disagree": 0, "fallback": 0, "no_geometry": 0, "patella": 0, "skipped_v1_ok": 0}\n    why_all = {}\n    for i, uid in enumerate(uids):\n        outp = os.path.join(a.out, study_key(uid) + ".json")\n        d = json.load(open(os.path.join(a.cache, "studies", study_key(uid) + ".json"), encoding="utf-8"))\n        if a.only_rework:\n            v1p = os.path.join(a.only_rework, study_key(uid) + ".json")\n            if os.path.exists(v1p) and plausible_v1(json.load(open(v1p)), d):\n                stats["skipped_v1_ok"] += 1\n                continue\n        elif os.path.exists(outp) and json.load(open(outp)).get("version") == 2:\n            continue\n        rec = bank.record(uid, d)\n        m = rec["method"] or ""\n        if m.startswith("skm_tea_joint"):\n            stats["ok"] += 1\n            if "avg" in m: stats["ok_avg"] += 1\n        elif m == "slab_prior":\n            stats["fallback"] += 1\n        else:\n            stats["no_geometry"] += 1\n        if rec["patella_lps"] is not None: stats["patella"] += 1\n        for w in rec["why"]:\n            k = w.split("(")[0]; why_all[k] = why_all.get(k, 0) + 1\n            if k == "disagree": stats["disagree"] += 1\n        json.dump(rec, open(outp, "w"), indent=0)\n        if (i + 1) % 100 == 0:\n            print(i + 1, stats, "%.0fs" % (time.time() - t0), flush=True)\n    print("done", stats, "why:", why_all, "%.0fs" % (time.time() - t0), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'side_common.py': '"""Image-based knee side check, shared by the training-cache fix and Kaggle inference.\n\nThe native cache stores coronal and axial series in a canonical frame where the study\'s side has already been applied:\nlateral should sit at +column for both knees (a right knee is mirrored as the last pixel operation, see\ntraining/dicom_geometry.flip_for_plane).  When the side is wrong, every coronal/axial image of the study is mirrored.\nside_inputs() turns one canonical native volume into a few normalised 2-D images; the classifier says whether lateral is\nat +column (correct orientation) or at -column (mirrored, i.e. the study\'s side is wrong)."""\nimport numpy as np\nimport cv2\n\nSIDE_SIZE = 160\nSIDE_FRACS = (0.4, 0.5, 0.6)\nSIDE_PLANES = ("Coronal", "Axial")\n\n\ndef side_inputs(vol):\n    """canonical native volume [n, H, W] (any dtype) -> uint8 [k, SIDE_SIZE, SIDE_SIZE], k <= 3 middle slices."""\n    n = int(vol.shape[0])\n    if n < 1:\n        return np.zeros((0, SIDE_SIZE, SIDE_SIZE), np.uint8)\n    idx = sorted({int(round(f * (n - 1))) for f in SIDE_FRACS})\n    out = []\n    for i in idx:\n        im = np.asarray(vol[i], np.float32)\n        lo, hi = np.percentile(im[::4, ::4], [1, 99])\n        im = np.clip((im - lo) / max(float(hi - lo), 1e-6), 0, 1)\n        im = cv2.resize(im, (SIDE_SIZE, SIDE_SIZE), interpolation=cv2.INTER_AREA)\n        out.append(np.rint(im * 255).astype(np.uint8))\n    return np.stack(out)\n\n\ndef build_side_net():\n    import timm, torch.nn as nn\n    net = timm.create_model("resnet34", pretrained=False, num_classes=1, in_chans=1)\n    return net\n'}

import os, sys, json, shutil, subprocess, zipfile, time, hashlib
from pathlib import Path
T0 = NB_T0   # [blend] the 8.3 h deadline counts from the notebook start (the Raptor stage ran first)
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], check=False)
subprocess.run(['df', '-h', '/dev/shm', '/kaggle/working', '/tmp'], check=False)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
TMP = Path('/kaggle/temp') if Path('/kaggle/temp').is_dir() else Path('/tmp')
TMP = TMP / 'probe'
TMP.mkdir(parents=True, exist_ok=True)

def find_mount(slug, required):
    cands = [INPUT / slug, INPUT / 'competitions' / slug]
    cands += list(INPUT.glob('datasets/*/' + slug)) + list(INPUT.glob('datasets/*/' + slug + '/*'))
    cands += list(INPUT.glob('*/' + slug)) + list(INPUT.glob(slug + '/*'))
    found = []
    for p in cands:
        if (p / required).is_file() and p not in found:
            found.append(p)
    if len(found) != 1:
        raise RuntimeError('Cannot resolve %s: %s' % (slug, found))
    return found[0]

CODE_SRC = find_mount(CODE_SLUG, 'receipt.json')
DATA = find_mount(COMPETITION, 'test.csv')

# mainline decode + rendering code: byte-identical to what built the training caches
CODE = TMP / 'mainline_code'
if (CODE_SRC / 'cloud_code.zip').is_file():
    with zipfile.ZipFile(CODE_SRC / 'cloud_code.zip') as z:
        z.extractall(CODE)
else:
    shutil.copytree(CODE_SRC / 'cloud_code', CODE, dirs_exist_ok=True)
for name, digest in CODE_SHA.items():
    assert hashlib.sha256((CODE / 'training' / name).read_bytes()).hexdigest() == digest, ('mainline module differs', name)

# pinned wheels (same versions as the runs that produced the training caches)
WHEELS = TMP / 'wheels'
WHEELS.mkdir(exist_ok=True)
if (CODE_SRC / 'dependencies.zip').is_file():
    with zipfile.ZipFile(CODE_SRC / 'dependencies.zip') as z:
        for n in z.namelist():
            if n.endswith('.whl'):
                (WHEELS / Path(n).name).write_bytes(z.read(n))
else:
    for p in CODE_SRC.rglob('*.whl'):
        shutil.copy(p, WHEELS / p.name)
wheels = sorted(str(p) for p in WHEELS.glob('*.whl'))
assert wheels, 'no wheels found'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', *wheels], check=True)

SRC = TMP / 'probe_src'
SRC.mkdir(exist_ok=True)
for name, text in SOURCES.items():
    (SRC / name).write_bytes(text.encode('utf-8'))
    assert hashlib.sha256((SRC / name).read_bytes()).hexdigest() == SOURCE_SHA[name], name
BUNDLE = TMP / 'bundle'
BUNDLE.mkdir(exist_ok=True)

M_SRC = find_mount(C_SLUG, 'manifest.json')
manifest = json.loads((M_SRC / 'manifest.json').read_text())
assert manifest['bundle_id'] == BUNDLE_ID, (manifest['bundle_id'], BUNDLE_ID)
for name in manifest['models'] + (manifest.get('models2') or []):
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('state_spec'):
    name = manifest['state_spec']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('side_model'):
    name = manifest['side_model']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
WORKER = 'compact_infer.py'
WORKER_ARGS = ['--backbone', manifest['backbone'], '--size', str(manifest['size']), '--crop', str(manifest['crop']),
               '--eval-stride', str(manifest.get('eval_stride', 3)), '--views', manifest.get('views', 'v1')]
if manifest.get('state_spec'):
    WORKER_ARGS += ['--state-spec', str(BUNDLE / manifest['state_spec'])]
if manifest.get('side_model'):
    # image side check: studies without one trustworthy laterality tag are re-rendered when the orientation classifier
    # says they are mirrored (same rule as the S5 training-cache fix)
    WORKER_ARGS += ['--side-model', str(BUNDLE / manifest['side_model']), '--side-threshold', str(manifest.get('side_threshold', 2.0)),
                    '--side-position-mm', str(manifest.get('side_position_mm', 20.0)), '--side-max-x', str(manifest.get('side_max_x', 0.0))]
WORKER_ARGS += ['--slices', str(manifest.get('slices', 3)), '--phys-mm', str(manifest.get('phys_mm', 0.0))]
if manifest.get('models2'):
    # second architecture on the same windows (one decode per study), per-class weights fixed in the bundle
    WORKER_ARGS += ['--models2', str(BUNDLE), '--arm2', manifest['arm2'], '--backbone2', manifest['backbone2'],
                    '--w2', ','.join(str(x) for x in manifest['w2'])]
if manifest.get('views') in ('v3k', 'v4'):
    # knee-centred views: the anchors are rebuilt from the test DICOM headers + the SKM-TEA checkpoint of the code dataset
    assert hashlib.sha256((CODE_SRC / 'anatomy.pt').read_bytes()).hexdigest() == manifest['anchor_weights_sha256'], 'anatomy.pt differs from the training anchor checkpoint'
    WORKER_ARGS += ['--anchor-weights', str(CODE_SRC / 'anatomy.pt')]
if manifest['backbone'] == 'ofl_l':
    # the fold models hold only the trained tail; rebuild the frozen part from the public OFL-L weights
    W_SRC = find_mount(W_SLUG, 'weight_parts.json')
    parts = json.loads((W_SRC / 'weight_parts.json').read_text())
    h = hashlib.sha256()
    with (BUNDLE / 'model.safetensors').open('wb') as dst:
        for item in parts['parts']:
            blob = (W_SRC / item['name']).read_bytes()
            assert len(blob) == item['bytes'] and hashlib.sha256(blob).hexdigest() == item['sha256'], item['name']
            dst.write(blob)
            h.update(blob)
    assert h.hexdigest() == parts['assembled_sha256'] == manifest['init_weights_sha256'], 'encoder weights differ'
    del blob
    WORKER_ARGS += ['--init', str(BUNDLE / 'model.safetensors'), '--freeze-blocks', str(manifest['freeze_blocks'])]
print('ASSETS_READY', str(CODE_SRC), str(M_SRC), str(DATA), 'seconds %.1f' % (time.time() - T0), flush=True)


In [ ]:

deadline = T0 + DEADLINE_HOURS * 3600
env = dict(os.environ, PYTHONPATH=str(CODE) + os.pathsep + os.environ.get('PYTHONPATH', ''), PYTHONDONTWRITEBYTECODE='1')
pred = TMP / 'prediction'
rc = subprocess.run([sys.executable, '-u', str(SRC / 'probe_dual.py'), '--data', str(DATA), '--split', 'test',
                     '--code', str(CODE), '--bundle', str(BUNDLE), '--arm', ARM, '--output', str(pred),
                     '--amp', 'auto', '--loader-workers', '2', '--deadline', str(deadline),
                     '--worker', WORKER, '--worker-args', json.dumps(WORKER_ARGS)], env=env).returncode
for log in sorted(pred.glob('worker*/log.txt')):
    shutil.copyfile(log, WORK / (log.parent.name + '_log.txt'))
if rc != 0:
    raise RuntimeError('controller failed with exit code %d' % rc)
shutil.copyfile(pred / 'submission.csv', WORK / 'ours_sub.csv')

if PROBE_CLASS:
    # single-class probe: every column except PROBE_CLASS is a constant, so public LB = (11 * 0.5 + AUC_c) / 12
    import pandas as pd
    _df = pd.read_csv(WORK / 'ours_sub.csv')
    assert PROBE_CLASS in _df.columns, PROBE_CLASS
    for _c in _df.columns:
        if _c not in ('StudyInstanceUID', PROBE_CLASS):
            _df[_c] = 0.5
    _df.to_csv(WORK / 'ours_sub.csv', index=False)
    print('SINGLE_CLASS_PROBE', PROBE_CLASS, 'other 11 columns set to 0.5', flush=True)
receipt = json.loads((pred / 'submission_receipt.json').read_text())
receipt['seconds_including_notebook_setup'] = time.time() - T0
receipt['bundle_id'] = BUNDLE_ID
receipt['arm'] = ARM
(WORK / 'submission_receipt.json').write_text(json.dumps(receipt, indent=1))
print('OURS_STAGE_COMPLETE', json.dumps({k: receipt.get(k) for k in ('status', 'arm', 'rows', 'scored', 'fallback_studies',
                                                                 'seconds', 'per_study_mean', 'gpus')}), flush=True)
import pandas as pd
display(pd.read_csv(WORK / 'ours_sub.csv').head())


In [ ]:
# Fork954 --s6-anchor: S6's own predictions (stage 2, unchanged, wrote ours_sub.csv) become the anchor submission.csv.
# Stage 3 (the 0.7 / 0.3 blend with the CoAtNet) is not in this notebook. A failed S6 stage has already raised above.
import pandas as _fork_pd0
_fork_ids = _fork_pd0.read_csv(DATA / 'test.csv')['StudyInstanceUID'].astype(str).tolist()
_fork_s6 = _fork_pd0.read_csv(WORK / 'ours_sub.csv', dtype={'StudyInstanceUID': str})
_fork_cols = (list(_fork_pd0.read_csv(DATA / 'sample_submission.csv', nrows=1).columns)
              if (DATA / 'sample_submission.csv').is_file() else list(_fork_s6.columns))
assert sorted(_fork_s6['StudyInstanceUID']) == sorted(_fork_ids), 'S6: rows differ from test.csv'
assert set(_fork_cols) <= set(_fork_s6.columns), 'S6: columns differ'
_fork_s6.set_index('StudyInstanceUID').loc[_fork_ids].reset_index()[_fork_cols].to_csv(WORK / 'submission.csv', index=False)
print('S6_ANCHOR_WRITTEN rows %d' % len(_fork_ids), flush=True)


## Fork arm — our members blended onto heliosli's S6 alone (OAI-free: no CoAtNet stage in this notebook)

The previous cell has written S6's predictions (stage 2 of the 0.954 notebook, unchanged) as `submission.csv`. This cell runs our own inference
pipeline (`src/kaggle_pipeline.py`, `MODE="infer"`, `INFER_MEMBERS = ["v15c", "v13b3"]`) as a subprocess on `cuda:0`, then
rank-blends per finding:

    final = rank_pct((1 - β) · rank_pct(anchor) + β · rank_pct(ours)),   β = 0.45

Outputs: `submission.csv` (β = 0.45) · `submission_fork_anchor_0954.csv` (exact anchor) ·
`submission_fork_beta*.csv` (β = (0.10, 0.30)) · `_ours.csv` · `ours_infer.log` · `fork_diagnostics.json`.

Fail-soft: if the anchor used more than 6.0 h, or our subprocess fails, times out or writes an invalid CSV,
`submission.csv` stays byte-identical to the anchor and the log says so. β = 0 is the anchor-only control.


In [ ]:
# Our inference pipeline (src/kaggle_pipeline.py, sed'd to MODE="infer"), zlib + base64.
# Built by src/build_fork949.py; pipeline sha256 c287ef990e0f623145f0d21bda2f54900d84e1a6ba92978580e1ffdf1d6e42d7
_FORK_PAYLOAD_SHA256 = 'c287ef990e0f623145f0d21bda2f54900d84e1a6ba92978580e1ffdf1d6e42d7'
_FORK_PAYLOAD = (
    '...'
)


In [ ]:
# ===================== FORK ARM: ours as a beta-blend on heliosli's S6 alone (OAI-free) =====================  
import base64 as _fork_b64, gc as _fork_gc, hashlib as _fork_hashlib, json as _fork_json, os as _fork_os
import shutil as _fork_shutil, signal as _fork_signal, subprocess as _fork_sp, sys as _fork_sys
import threading as _fork_thr, time as _fork_time, traceback as _fork_tb, zlib as _fork_zlib
from pathlib import Path as _ForkPath
import numpy as _fork_np, pandas as _fork_pd

_FORK_WORK = _ForkPath('/kaggle/working')
_FORK_FINAL = _FORK_WORK / 'submission.csv'
_FORK_ANCHOR = _FORK_WORK / '_anchor_0954.csv'
_FORK_ANCHOR_PUB = _FORK_WORK / 'submission_fork_anchor_0954.csv'
_FORK_OURS = _FORK_WORK / '_ours.csv'
_FORK_SCRIPT = _FORK_WORK / 'ours_infer.py'
_FORK_LOG = _FORK_WORK / 'ours_infer.log'
_FORK_DIAG = _FORK_WORK / 'fork_diagnostics.json'
_FORK_LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
                'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_FORK_MEMBERS = ["v15c", "v13b3"]
_FORK_BETA = 0.45
_FORK_BETAS_DIAG = (0.10, 0.30)
_FORK_GATE_H = 6.0
_FORK_HARD_STOP_H = 8.4
_FORK_SEC_PER_STUDY = 5.5


class _ForkControl(Exception):   # beta 0: the anchor-only control -- our arm is deliberately not run
    pass


def _fork_log(msg):
    print(f'[fork {(_fork_time.time() - _FORK_T0) / 3600:5.2f}h] {msg}', flush=True)


# --- pure ---
def _fork_rankpct(values):
    return _fork_pd.DataFrame(_fork_np.asarray(values, _fork_np.float64)).rank(
        method='average', pct=True).to_numpy(_fork_np.float64)


def _fork_blend(anchor, ours, beta):
    a = _fork_rankpct(anchor[_FORK_LABELS].to_numpy())
    o = _fork_rankpct(ours[_FORK_LABELS].to_numpy())
    out = anchor.copy()
    out[_FORK_LABELS] = _fork_rankpct((1.0 - float(beta)) * a + float(beta) * o)
    return out


def _fork_spearman(anchor, ours):
    return {lab: float(anchor[lab].corr(ours[lab], method='spearman')) for lab in _FORK_LABELS}


def _fork_validate_ours(frame, uids):
    if frame.columns.tolist() != ['StudyInstanceUID', *_FORK_LABELS]:
        raise ValueError(f'ours: schema {frame.columns.tolist()}')
    frame = frame.copy()
    frame['StudyInstanceUID'] = frame['StudyInstanceUID'].astype(str)
    if frame.StudyInstanceUID.duplicated().any() or set(frame.StudyInstanceUID) != set(map(str, uids)):
        raise ValueError('ours: UID set / duplicates differ from the anchor')
    v = frame[_FORK_LABELS].to_numpy(_fork_np.float64)
    if not _fork_np.isfinite(v).all() or v.min() < 0 or v.max() > 1:
        raise ValueError('ours: non-finite or out-of-range values')
    if len(frame) > 3 and int((v.std(0) < 1e-9).sum()) > len(_FORK_LABELS) // 2:
        raise ValueError('ours: mostly constant columns')
    return frame.set_index('StudyInstanceUID').loc[[str(u) for u in uids]].reset_index()
# --- end pure ---


def _fork_sha(path):
    return _fork_hashlib.sha256(_ForkPath(path).read_bytes()).hexdigest()


def _fork_check_anchor(frame):
    if sorted(frame.columns.tolist()) != sorted(['StudyInstanceUID', *_FORK_LABELS]):
        raise ValueError(f'anchor: schema {frame.columns.tolist()}')
    if frame.StudyInstanceUID.duplicated().any():
        raise ValueError('anchor: duplicate UIDs')
    if not _fork_np.isfinite(frame[_FORK_LABELS].to_numpy(_fork_np.float64)).all():
        raise ValueError('anchor: non-finite values')
    return frame


def _fork_write_csv(frame, path):
    tmp = path.with_suffix('.csv.tmp')
    frame.to_csv(tmp, index=False)
    _fork_os.replace(tmp, path)


def _fork_free_gpu():
    _fork_gc.collect()
    try:
        import torch as _fork_torch
        if _fork_torch.cuda.is_available():
            for _i in range(_fork_torch.cuda.device_count()):
                with _fork_torch.cuda.device(_i):
                    _fork_torch.cuda.empty_cache()
            _fork_log('gpu free after the anchor: ' + ', '.join(
                f'cuda:{_i} {_fork_torch.cuda.mem_get_info(_i)[0] / 2**30:.1f} GiB'
                for _i in range(_fork_torch.cuda.device_count())))
    except Exception as _e:
        _fork_log(f'gpu cleanup skipped: {type(_e).__name__}: {_e}')


def _fork_run_ours(timeout_s):
    raw = _fork_zlib.decompress(_fork_b64.b64decode(_FORK_PAYLOAD)).decode('utf-8')
    if _fork_hashlib.sha256(raw.encode('utf-8')).hexdigest() != _FORK_PAYLOAD_SHA256:
        raise RuntimeError('payload sha256 mismatch')
    compile(raw, str(_FORK_SCRIPT), 'exec')
    _FORK_SCRIPT.write_text(raw, encoding='utf-8')
    env = dict(_fork_os.environ)
    env.update(PYTHONUTF8='1', PYTHONUNBUFFERED='1', RSNA_WORKERS='4', CUDA_VISIBLE_DEVICES='0',
               HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
    proc = _fork_sp.Popen([_fork_sys.executable, str(_FORK_SCRIPT)], cwd=str(_FORK_WORK), env=env,
                          stdout=_fork_sp.PIPE, stderr=_fork_sp.STDOUT, text=True, errors='replace',
                          bufsize=1, start_new_session=True)
    killed = []

    def _kill():
        killed.append(True)
        for sig, wait in ((_fork_signal.SIGTERM, 30), (_fork_signal.SIGKILL, 10)):
            try:
                _fork_os.killpg(proc.pid, sig)
                proc.wait(timeout=wait)
                return
            except Exception:
                pass

    timer = _fork_thr.Timer(max(1.0, float(timeout_s)), _kill)
    timer.start()
    try:
        with _FORK_LOG.open('w', encoding='utf-8') as handle:
            for line in proc.stdout:
                handle.write(line)
                print('[ours] ' + line.rstrip('\n'), flush=True)
        rc = proc.wait()
    finally:
        timer.cancel()
    if killed:
        raise TimeoutError(f'ours_infer.py killed at the {_FORK_HARD_STOP_H} h hard stop')
    return rc


_fork_status, _fork_reason = 'anchor', ''
_fork_diag = {'anchor': 'S6 alone (heliosli compact model; no OAI-trained stage)', 'members': list(_FORK_MEMBERS), 'beta': _FORK_BETA,
              'betas_diag': list(_FORK_BETAS_DIAG)}
_fork_anchor = _fork_check_anchor(_fork_pd.read_csv(_FORK_FINAL, dtype={'StudyInstanceUID': str}))
_fork_shutil.copyfile(_FORK_FINAL, _FORK_ANCHOR)
_fork_shutil.copyfile(_FORK_FINAL, _FORK_ANCHOR_PUB)
_fork_anchor_sha = _fork_sha(_FORK_ANCHOR)
_fork_log(f'anchor saved -> {_FORK_ANCHOR.name} rows={len(_fork_anchor)} sha256={_fork_anchor_sha}')
try:
    if _FORK_BETA <= 0.0:
        raise _ForkControl('beta 0: anchor-only control -- our arm is not run, submission.csv = the exact anchor')
    _fork_free_gpu()
    _fork_elapsed = _fork_time.time() - _FORK_T0
    _fork_est = len(_fork_anchor) * _FORK_SEC_PER_STUDY + 600
    if _fork_elapsed > _FORK_GATE_H * 3600:
        raise TimeoutError(f'anchor took {_fork_elapsed / 3600:.2f} h > gate {_FORK_GATE_H} h')
    if _fork_elapsed + _fork_est > _FORK_HARD_STOP_H * 3600:
        raise TimeoutError(f'estimated ours {_fork_est / 60:.0f} min would pass the {_FORK_HARD_STOP_H} h hard stop')
    _fork_log(f'launching ours ({", ".join(_FORK_MEMBERS)}) on cuda:0; budget '
              f'{(_FORK_HARD_STOP_H * 3600 - _fork_elapsed) / 60:.0f} min')
    _fork_t = _fork_time.time()
    _fork_rc = _fork_run_ours(_FORK_HARD_STOP_H * 3600 - _fork_elapsed)
    _fork_diag['subprocess'] = {'returncode': int(_fork_rc), 'seconds': _fork_time.time() - _fork_t}
    if _fork_rc != 0:
        raise RuntimeError(f'ours_infer.py exited {_fork_rc} (see ours_infer.log)')
    if not _FORK_FINAL.is_file():
        raise FileNotFoundError('ours_infer.py wrote no submission.csv')
    _fork_os.replace(_FORK_FINAL, _FORK_OURS)
    _fork_ours = _fork_validate_ours(_fork_pd.read_csv(_FORK_OURS, dtype={'StudyInstanceUID': str}),
                                     _fork_anchor.StudyInstanceUID.tolist())
    _fork_diag['spearman_ours_vs_anchor'] = _fork_spearman(_fork_anchor, _fork_ours)
    _fork_diag['spearman_mean'] = float(_fork_np.nanmean(list(_fork_diag['spearman_ours_vs_anchor'].values())))
    _fork_log('spearman(ours, anchor) per finding: '
              + str({k: round(v, 3) for k, v in _fork_diag['spearman_ours_vs_anchor'].items()}))
    for _b in _FORK_BETAS_DIAG:
        _fork_write_csv(_fork_blend(_fork_anchor, _fork_ours, _b),
                        _FORK_WORK / f'submission_fork_beta{int(round(_b * 100)):03d}.csv')
    _fork_main = _fork_blend(_fork_anchor, _fork_ours, _FORK_BETA)
    if not _fork_np.isfinite(_fork_main[_FORK_LABELS].to_numpy(_fork_np.float64)).all():
        raise ValueError('blend: non-finite values')
    _fork_write_csv(_fork_main, _FORK_FINAL)
    _fork_status = f'beta{_FORK_BETA:.2f}'
except _ForkControl as _fork_exc:
    _fork_status, _fork_reason = 'anchor_control', str(_fork_exc)
    _fork_log('ANCHOR-ONLY CONTROL -> ' + _fork_reason)
except BaseException as _fork_exc:
    _fork_reason = f'{type(_fork_exc).__name__}: {str(_fork_exc)[:800]}'
    _fork_log('OUR ARM FAILED/SKIPPED -> anchor retained: ' + _fork_reason)
    _fork_tb.print_exc()
finally:
    if _fork_status == 'anchor' or not _FORK_FINAL.is_file():
        _fork_shutil.copyfile(_FORK_ANCHOR, _FORK_FINAL)
        _fork_status = 'anchor'
    _fork_final_sha = _fork_sha(_FORK_FINAL)
    if _fork_status in ('anchor', 'anchor_control') and _fork_final_sha != _fork_anchor_sha:
        _fork_shutil.copyfile(_FORK_ANCHOR, _FORK_FINAL)
        _fork_final_sha = _fork_sha(_FORK_FINAL)
    _fork_diag.update(status=_fork_status, reason=_fork_reason, anchor_sha256=_fork_anchor_sha,
                      submission_sha256=_fork_final_sha, elapsed_h=(_fork_time.time() - _FORK_T0) / 3600)
    _FORK_DIAG.write_text(_fork_json.dumps(_fork_diag, indent=1), encoding='utf-8')
    _fork_log(f'FINAL submission.csv = {_fork_status} sha256={_fork_final_sha}')
